In [ ]:

from pathlib import Path
import os

# Identify the project directory
project_dir = Path.cwd().resolve()

# If the notebook starts from the notebooks folder, move one level up
if project_dir.name.lower() == "notebooks":
    project_dir = project_dir.parent

# Set project directory as working directory
os.chdir(project_dir)

# Confirm project location
print("Project directory:", Path.cwd())

# Confirm required folders
required_folders = [
    "data/raw",
    "data/processed",
    "data/final",
    "notebooks",
    "sql",
    "reports",
    "power_bi",
    "documentation"
]

for folder in required_folders:
    (project_dir / folder).mkdir(parents=True, exist_ok=True)

print("Project folder structure verified.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Define data output directory
raw_data_dir = project_dir / "data" / "raw"

print("Libraries imported successfully.")


In [ ]:

category_distribution = {
    "Electronics": 100,
    "Home & Kitchen": 200,
    "Clothing": 500,
    "Beauty": 300
}

demand_distribution = {
    "Stable": 330,
    "Seasonal": 275,
    "Growing": 220,
    "Declining": 165,
    "Intermittent": 110
}

print("Total categories:", len(category_distribution))
print("Total SKUs:", sum(category_distribution.values()))
print("Total demand patterns:", sum(demand_distribution.values()))


In [ ]:

# Generate category assignments
category_list = []

for category, count in category_distribution.items():
    category_list.extend([category] * count)

np.random.shuffle(category_list)

# Generate demand-pattern assignments
demand_pattern_list = []

for pattern, count in demand_distribution.items():
    demand_pattern_list.extend([pattern] * count)

np.random.shuffle(demand_pattern_list)

# Create product catalogue
products_df = pd.DataFrame({
    "SKU_ID": [f"SKU-{i:04d}" for i in range(1, 1101)],
    "Category": category_list,
    "Demand_Pattern": demand_pattern_list
})

# Review product catalogue
display(products_df.head(10))

print("\nCategory distribution:")
display(products_df["Category"].value_counts())

print("\nDemand pattern distribution:")
display(products_df["Demand_Pattern"].value_counts())


In [ ]:
# Define typical selling price ranges by category
price_ranges = {
    "Electronics": (1500, 30000),
    "Home & Kitchen": (300, 8000),
    "Clothing": (400, 5000),
    "Beauty": (150, 3000)
}

# Generate product attributes
products_df["Product_Name"] = [
    f"{category.split()[0]} Product {i:04d}"
    for i, category in enumerate(products_df["Category"], start=1)
]

products_df["Unit_Selling_Price"] = [
    round(np.random.uniform(*price_ranges[category]), 2)
    for category in products_df["Category"]
]

# Assume purchase cost is between 55% and 75% of selling price
products_df["Unit_Purchase_Cost"] = (
    products_df["Unit_Selling_Price"] *
    np.random.uniform(0.55, 0.75, len(products_df))
).round(2)

# Review the product catalogue
display(products_df.head(10))

print("Total SKUs:", len(products_df))
print("Missing values:", products_df.isnull().sum().sum())

In [ ]:
suppliers_df = pd.DataFrame({
    "Supplier_ID": [f"SUP-{i:02d}" for i in range(1, 11)],
    "Supplier_Name": [f"Supplier {i:02d}" for i in range(1, 11)],
    "Lead_Time_Days": np.random.randint(7, 36, 10),
    "Supplier_Reliability": np.random.choice(
        ["High", "Medium", "Low"],
        size=10,
        p=[0.3, 0.5, 0.2]
    )
})

display(suppliers_df)

print("Total suppliers:", len(suppliers_df))
print("Minimum lead time:", suppliers_df["Lead_Time_Days"].min())
print("Maximum lead time:", suppliers_df["Lead_Time_Days"].max())

In [ ]:
# Assign one preferred supplier to every SKU
products_df["Supplier_ID"] = np.random.choice(
    suppliers_df["Supplier_ID"],
    size=len(products_df)
)

# Create the product-supplier mapping
product_supplier_df = products_df[
    ["SKU_ID", "Supplier_ID"]
].copy()

product_supplier_df["Supplier_Role"] = "Preferred"

# Review the mapping
display(product_supplier_df.head(10))

print("Total SKU mappings:", len(product_supplier_df))
print("Unique SKUs mapped:", product_supplier_df["SKU_ID"].nunique())
print("\nProducts assigned to each supplier:")
display(
    product_supplier_df["Supplier_ID"].value_counts().sort_index()
)

In [ ]:
sales_dates = pd.date_range(
    start="2024-01-01",
    end="2025-12-31",
    freq="D"
)

print("Start date:", sales_dates.min())
print("End date:", sales_dates.max())
print("Total calendar days:", len(sales_dates))

In [ ]:
# Define base daily demand ranges by category
demand_ranges = {
    "Electronics": (1, 5),
    "Home & Kitchen": (1, 8),
    "Clothing": (1, 12),
    "Beauty": (1, 10)
}

# Assign a base daily demand to each SKU
products_df["Base_Daily_Demand"] = [
    np.random.randint(
        demand_ranges[category][0],
        demand_ranges[category][1] + 1
    )
    for category in products_df["Category"]
]

display(
    products_df[
        ["SKU_ID", "Category", "Demand_Pattern", "Base_Daily_Demand"]
    ].head(10)
)

print("Minimum base demand:", products_df["Base_Daily_Demand"].min())
print("Maximum base demand:", products_df["Base_Daily_Demand"].max())
print("Missing base demand:", products_df["Base_Daily_Demand"].isna().sum())

In [ ]:
# Prepare SKU-level attributes for faster demand generation
sku_attributes = products_df.set_index("SKU_ID")

# Calendar features
calendar_df = pd.DataFrame({"Date": sales_dates})
calendar_df["Month"] = calendar_df["Date"].dt.month
calendar_df["Year"] = calendar_df["Date"].dt.year

# Create seasonal demand multiplier
calendar_df["Seasonal_Multiplier"] = 1.0

# Festive and year-end demand increases
calendar_df.loc[
    calendar_df["Month"].isin([10, 11]), "Seasonal_Multiplier"
] = 1.50

calendar_df.loc[
    calendar_df["Month"].isin([12]), "Seasonal_Multiplier"
] = 1.25

# Generate sales records
sales_records = []

for _, product in products_df.iterrows():

    sku = product["SKU_ID"]
    category = product["Category"]
    pattern = product["Demand_Pattern"]
    base_demand = product["Base_Daily_Demand"]

    for date in sales_dates:

        days_elapsed = (date - sales_dates[0]).days

        # Start with base demand
        expected_demand = base_demand

        # Apply demand pattern
        if pattern == "Growing":
            expected_demand *= (1 + 0.40 * days_elapsed / 730)

        elif pattern == "Declining":
            expected_demand *= (1 - 0.40 * days_elapsed / 730)

        elif pattern == "Seasonal":
            if date.month in [10, 11]:
                expected_demand *= 1.8
            elif date.month == 12:
                expected_demand *= 1.4

        elif pattern == "Intermittent":
            if np.random.random() < 0.75:
                expected_demand = 0

        # Apply category-level seasonal effects
        if category == "Electronics" and date.month in [10, 11]:
            expected_demand *= 1.25

        elif category == "Clothing" and date.month in [6, 7, 10, 11]:
            expected_demand *= 1.20

        elif category == "Home & Kitchen" and date.month in [10, 11, 12]:
            expected_demand *= 1.20

        elif category == "Beauty" and date.month in [10, 11, 12]:
            expected_demand *= 1.15

        # Add occasional promotional spikes
        if np.random.random() < 0.03:
            expected_demand *= 1.5

        # Add random daily variation
        expected_demand *= np.random.uniform(0.70, 1.30)

        # Generate non-negative daily sales
        units_sold = max(0, np.random.poisson(expected_demand))

        sales_records.append({
            "Date": date,
            "SKU_ID": sku,
            "Units_Sold": units_sold
        })

# Create daily sales dataset
daily_sales_df = pd.DataFrame(sales_records)

display(daily_sales_df.head(10))

print("Total sales records:", len(daily_sales_df))
print("Unique SKUs:", daily_sales_df["SKU_ID"].nunique())
print("Date range:", daily_sales_df["Date"].min(), "to", daily_sales_df["Date"].max())
print("Total units sold:", daily_sales_df["Units_Sold"].sum())
print("Zero-sales records:", (daily_sales_df["Units_Sold"] == 0).sum())

In [ ]:
# Add product category to daily sales
daily_sales_df = daily_sales_df.merge(
    products_df[["SKU_ID", "Category"]],
    on="SKU_ID",
    how="left"
)

# Summarise sales by category and year
daily_sales_df["Year"] = daily_sales_df["Date"].dt.year

sales_summary = (
    daily_sales_df
    .groupby(["Year", "Category"], as_index=False)
    .agg(
        Total_Units_Sold=("Units_Sold", "sum"),
        Average_Daily_Units=("Units_Sold", "mean"),
        Active_SKUs=("SKU_ID", "nunique")
    )
)

display(sales_summary)

print("Total units across summary:", sales_summary["Total_Units_Sold"].sum())
print("Missing categories:", daily_sales_df["Category"].isna().sum())

In [ ]:
# Save master datasets to the raw data folder
products_df.to_csv(
    raw_data_dir / "Products.csv",
    index=False
)

suppliers_df.to_csv(
    raw_data_dir / "Suppliers.csv",
    index=False
)

product_supplier_df.to_csv(
    raw_data_dir / "Product_Supplier_Mapping.csv",
    index=False
)

daily_sales_df.to_csv(
    raw_data_dir / "Daily_Sales.csv",
    index=False
)

print("Master datasets saved successfully.")

for file in raw_data_dir.glob("*.csv"):
    print(file.name, "-", round(file.stat().st_size / (1024 * 1024), 2), "MB")

In [ ]:
# Preserve generated demand before applying inventory constraints
daily_sales_df = daily_sales_df.rename(
    columns={"Units_Sold": "Potential_Demand"}
)

# Review the updated dataset
display(daily_sales_df.head(10))

print("Total records:", len(daily_sales_df))
print("Total potential demand:", daily_sales_df["Potential_Demand"].sum())
print("Missing potential demand:", daily_sales_df["Potential_Demand"].isna().sum())

In [ ]:
# Combine product details with supplier lead times
sku_planning_df = products_df.merge(
    suppliers_df[
        ["Supplier_ID", "Lead_Time_Days", "Supplier_Reliability"]
    ],
    on="Supplier_ID",
    how="left"
)

# Assign opening inventory for January 2024
opening_stock_days = np.random.randint(
    15, 31, size=len(sku_planning_df)
)

sku_planning_df["Opening_Inventory_2024"] = (
    sku_planning_df["Base_Daily_Demand"] * opening_stock_days
)

# Review planning master
display(
    sku_planning_df[
        [
            "SKU_ID",
            "Category",
            "Base_Daily_Demand",
            "Supplier_ID",
            "Lead_Time_Days",
            "Opening_Inventory_2024"
        ]
    ].head(10)
)

print("Total SKUs:", len(sku_planning_df))
print(
    "Missing supplier details:",
    sku_planning_df["Lead_Time_Days"].isna().sum()
)
print(
    "Total opening inventory:",
    sku_planning_df["Opening_Inventory_2024"].sum()
)

In [ ]:
# Generate monthly purchase orders for each SKU
po_records = []

po_counter = 1

purchase_months = pd.date_range(
    start="2024-01-01",
    end="2025-12-01",
    freq="MS"
)

for _, product in sku_planning_df.iterrows():

    for month_start in purchase_months:

        # Vary purchase order date within the first 10 days
        order_day = np.random.randint(1, 11)

        po_date = month_start + pd.Timedelta(days=order_day - 1)

        # Calculate order quantity based on base daily demand
        coverage_days = np.random.randint(25, 41)

        order_quantity = (
            product["Base_Daily_Demand"] * coverage_days
        )

        expected_delivery_date = (
            po_date +
            pd.Timedelta(days=int(product["Lead_Time_Days"]))
        )

        po_records.append({
            "PO_ID": f"PO-{po_counter:06d}",
            "SKU_ID": product["SKU_ID"],
            "Supplier_ID": product["Supplier_ID"],
            "PO_Date": po_date,
            "Order_Quantity": order_quantity,
            "Expected_Delivery_Date": expected_delivery_date
        })

        po_counter += 1

# Create purchase order dataset
purchase_orders_df = pd.DataFrame(po_records)

# Review purchase orders
display(purchase_orders_df.head(10))

print("Total purchase orders:", len(purchase_orders_df))
print("Unique SKUs:", purchase_orders_df["SKU_ID"].nunique())
print("Total ordered units:", purchase_orders_df["Order_Quantity"].sum())
print(
    "Purchase order date range:",
    purchase_orders_df["PO_Date"].min(),
    "to",
    purchase_orders_df["PO_Date"].max()
)

In [ ]:
# Generate purchase order receipt history
receipt_records = []

receipt_counter = 1
simulation_end_date = pd.Timestamp("2025-12-31")

for _, po in purchase_orders_df.iterrows():

    expected_date = po["Expected_Delivery_Date"]
    order_qty = int(po["Order_Quantity"])

    # Select delivery scenario
    scenario = np.random.choice(
        ["Full", "Partial", "No Delivery"],
        p=[0.80, 0.17, 0.03]
    )

    if scenario == "No Delivery":
        continue

    # Actual first delivery date: early or delayed
    first_receipt_date = (
        expected_date +
        pd.Timedelta(days=np.random.randint(-5, 11))
    )

    if first_receipt_date > simulation_end_date:
        continue

    if scenario == "Full":

        receipt_records.append({
            "Receipt_ID": f"REC-{receipt_counter:07d}",
            "PO_ID": po["PO_ID"],
            "SKU_ID": po["SKU_ID"],
            "Receipt_Date": first_receipt_date,
            "Received_Quantity": order_qty
        })

        receipt_counter += 1

    elif scenario == "Partial":

        # Split order into two deliveries
        first_qty = int(
            order_qty * np.random.uniform(0.50, 0.80)
        )

        remaining_qty = order_qty - first_qty

        receipt_records.append({
            "Receipt_ID": f"REC-{receipt_counter:07d}",
            "PO_ID": po["PO_ID"],
            "SKU_ID": po["SKU_ID"],
            "Receipt_Date": first_receipt_date,
            "Received_Quantity": first_qty
        })

        receipt_counter += 1

        # Schedule the remaining quantity for a later delivery
        second_receipt_date = (
            first_receipt_date +
            pd.Timedelta(days=np.random.randint(5, 16))
        )

        if second_receipt_date <= simulation_end_date:

            receipt_records.append({
                "Receipt_ID": f"REC-{receipt_counter:07d}",
                "PO_ID": po["PO_ID"],
                "SKU_ID": po["SKU_ID"],
                "Receipt_Date": second_receipt_date,
                "Received_Quantity": remaining_qty
            })

            receipt_counter += 1

# Create receipt dataset
purchase_order_receipts_df = pd.DataFrame(receipt_records)

# Review receipt history
display(purchase_order_receipts_df.head(10))

print("Total receipt records:", len(purchase_order_receipts_df))
print(
    "Unique purchase orders with receipts:",
    purchase_order_receipts_df["PO_ID"].nunique()
)
print(
    "Total received units:",
    purchase_order_receipts_df["Received_Quantity"].sum()
)
print(
    "Receipt date range:",
    purchase_order_receipts_df["Receipt_Date"].min(),
    "to",
    purchase_order_receipts_df["Receipt_Date"].max()
)

In [ ]:
# Summarise receipts against each purchase order
receipt_summary = (
    purchase_order_receipts_df
    .groupby("PO_ID", as_index=False)
    .agg(
        Total_Received_Quantity=("Received_Quantity", "sum"),
        First_Receipt_Date=("Receipt_Date", "min"),
        Last_Receipt_Date=("Receipt_Date", "max"),
        Receipt_Count=("Receipt_ID", "count")
    )
)

# Merge receipt summary with purchase orders
purchase_orders_df = purchase_orders_df.merge(
    receipt_summary,
    on="PO_ID",
    how="left"
)

# Replace missing receipt quantities with zero
purchase_orders_df["Total_Received_Quantity"] = (
    purchase_orders_df["Total_Received_Quantity"].fillna(0)
)

purchase_orders_df["Outstanding_Quantity"] = (
    purchase_orders_df["Order_Quantity"] -
    purchase_orders_df["Total_Received_Quantity"]
)

# Assign purchase order status
purchase_orders_df["PO_Status"] = np.where(
    purchase_orders_df["Outstanding_Quantity"] == 0,
    "Closed",
    "Open"
)

# Review reconciled purchase orders
display(purchase_orders_df.head(10))

print("\nPurchase order status:")
display(purchase_orders_df["PO_Status"].value_counts())

print(
    "Total ordered quantity:",
    purchase_orders_df["Order_Quantity"].sum()
)

print(
    "Total received quantity:",
    purchase_orders_df["Total_Received_Quantity"].sum()
)

print(
    "Total outstanding quantity:",
    purchase_orders_df["Outstanding_Quantity"].sum()
)

print(
    "Orders with negative outstanding quantity:",
    (purchase_orders_df["Outstanding_Quantity"] < 0).sum()
)

In [ ]:
# Prepare daily receipts by SKU and receipt date
daily_receipts_df = (
    purchase_order_receipts_df
    .groupby(["SKU_ID", "Receipt_Date"], as_index=False)
    ["Received_Quantity"].sum()
)

# Create a lookup of opening inventory by SKU
opening_inventory_lookup = sku_planning_df.set_index(
    "SKU_ID"
)["Opening_Inventory_2024"].to_dict()

# Create a lookup of receipts by SKU
receipt_lookup = {
    sku: group.set_index("Receipt_Date")["Received_Quantity"]
    for sku, group in daily_receipts_df.groupby("SKU_ID")
}

# Sort daily demand records
daily_sales_sim_df = (
    daily_sales_df
    .sort_values(["SKU_ID", "Date"])
    .reset_index(drop=True)
    .copy()
)

# Prepare output columns
daily_sales_sim_df["Opening_Inventory"] = 0
daily_sales_sim_df["Units_Received"] = 0
daily_sales_sim_df["Units_Sold"] = 0
daily_sales_sim_df["Closing_Inventory"] = 0
daily_sales_sim_df["In_Stock_Flag"] = False

# Internal simulation field
daily_sales_sim_df["Unfulfilled_Demand"] = 0

# Simulate inventory and sales SKU by SKU
for sku, group in daily_sales_sim_df.groupby("SKU_ID", sort=False):

    positions = group.index.to_numpy()

    potential_demand = group["Potential_Demand"].to_numpy()

    if sku in receipt_lookup:
        receipts = (
            receipt_lookup[sku]
            .reindex(group["Date"])
            .fillna(0)
            .to_numpy(dtype=int)
        )
    else:
        receipts = np.zeros(len(group), dtype=int)

    stock = int(opening_inventory_lookup[sku])

    opening_stock = np.zeros(len(group), dtype=int)
    actual_sales = np.zeros(len(group), dtype=int)
    closing_stock = np.zeros(len(group), dtype=int)
    in_stock = np.zeros(len(group), dtype=bool)
    unfulfilled = np.zeros(len(group), dtype=int)

    for i in range(len(group)):

        opening_stock[i] = stock

        available_stock = stock + receipts[i]

        # Count as in-stock if inventory was available
        # at any point when the day began
        in_stock[i] = available_stock > 0

        # Actual sales cannot exceed available inventory
        actual_sales[i] = min(
            potential_demand[i],
            available_stock
        )

        # Internal simulation measure
        unfulfilled[i] = (
            potential_demand[i] - actual_sales[i]
        )

        # Calculate closing inventory
        stock = available_stock - actual_sales[i]

        closing_stock[i] = stock

    # Assign simulated results
    daily_sales_sim_df.loc[positions, "Opening_Inventory"] = opening_stock
    daily_sales_sim_df.loc[positions, "Units_Received"] = receipts
    daily_sales_sim_df.loc[positions, "Units_Sold"] = actual_sales
    daily_sales_sim_df.loc[positions, "Closing_Inventory"] = closing_stock
    daily_sales_sim_df.loc[positions, "In_Stock_Flag"] = in_stock
    daily_sales_sim_df.loc[positions, "Unfulfilled_Demand"] = unfulfilled

# Review the simulated dataset
display(daily_sales_sim_df.head(10))

print("Total records:", len(daily_sales_sim_df))
print("Total potential demand:", daily_sales_sim_df["Potential_Demand"].sum())
print("Total actual sales:", daily_sales_sim_df["Units_Sold"].sum())
print("Total unfulfilled demand:", daily_sales_sim_df["Unfulfilled_Demand"].sum())
print("In-stock days:", daily_sales_sim_df["In_Stock_Flag"].sum())
print("Out-of-stock days:", (~daily_sales_sim_df["In_Stock_Flag"]).sum())
print("Total closing inventory:", daily_sales_sim_df["Closing_Inventory"].sum())

In [ ]:
# Check daily inventory balance:
# Opening inventory + receipts - actual sales = closing inventory

daily_sales_sim_df["Calculated_Closing_Inventory"] = (
    daily_sales_sim_df["Opening_Inventory"]
    + daily_sales_sim_df["Units_Received"]
    - daily_sales_sim_df["Units_Sold"]
)

balance_errors = (
    daily_sales_sim_df["Calculated_Closing_Inventory"]
    != daily_sales_sim_df["Closing_Inventory"]
).sum()

# Check that each day's opening inventory matches
# the previous day's closing inventory for each SKU
daily_sales_sim_df["Previous_Day_Closing"] = (
    daily_sales_sim_df.groupby("SKU_ID")["Closing_Inventory"].shift(1)
)

continuity_errors = (
    daily_sales_sim_df["Opening_Inventory"]
    != daily_sales_sim_df["Previous_Day_Closing"]
).sum()

# Calculate actual year-end inventory
year_end_inventory = (
    daily_sales_sim_df
    .sort_values(["SKU_ID", "Date"])
    .groupby("SKU_ID")
    .tail(1)["Closing_Inventory"]
    .sum()
)

# Overall inventory reconciliation
total_opening_inventory = sku_planning_df["Opening_Inventory_2024"].sum()
total_receipts = daily_sales_sim_df["Units_Received"].sum()
total_actual_sales = daily_sales_sim_df["Units_Sold"].sum()

reconciled_year_end_inventory = (
    total_opening_inventory
    + total_receipts
    - total_actual_sales
)

print("Daily balance errors:", balance_errors)
print("Inventory continuity errors:", continuity_errors)

print("\nInventory reconciliation:")
print("Opening inventory:", total_opening_inventory)
print("Total receipts:", total_receipts)
print("Actual sales:", total_actual_sales)
print("Calculated year-end inventory:", reconciled_year_end_inventory)
print("Actual year-end inventory:", year_end_inventory)

print(
    "\nYear-end reconciliation difference:",
    reconciled_year_end_inventory - year_end_inventory
)

In [ ]:
# Check continuity between consecutive days for each SKU
daily_sales_sim_df["Previous_Day_Closing"] = (
    daily_sales_sim_df
    .groupby("SKU_ID")["Closing_Inventory"]
    .shift(1)
)

# Exclude the first day for each SKU
continuity_check_df = daily_sales_sim_df[
    daily_sales_sim_df["Previous_Day_Closing"].notna()
]

continuity_errors = (
    continuity_check_df["Opening_Inventory"]
    != continuity_check_df["Previous_Day_Closing"]
).sum()

print("Continuity records checked:", len(continuity_check_df))
print("Inventory continuity errors:", continuity_errors)

In [ ]:
# Prepare client-facing daily sales dataset
daily_sales_final_df = daily_sales_sim_df[
    [
        "Date",
        "SKU_ID",
        "Category",
        "Year",
        "Opening_Inventory",
        "Units_Received",
        "Units_Sold",
        "Closing_Inventory",
        "In_Stock_Flag"
    ]
].copy()

# Save the datasets
products_df.to_csv(raw_data_dir / "Products.csv", index=False)

suppliers_df.to_csv(raw_data_dir / "Suppliers.csv", index=False)

product_supplier_df.to_csv(
    raw_data_dir / "Product_Supplier_Mapping.csv",
    index=False
)

daily_sales_final_df.to_csv(
    raw_data_dir / "Daily_Sales.csv",
    index=False
)

purchase_orders_df.to_csv(
    raw_data_dir / "Purchase_Orders.csv",
    index=False
)

purchase_order_receipts_df.to_csv(
    raw_data_dir / "Purchase_Order_Receipts.csv",
    index=False
)

print("All six datasets saved successfully.")

print("\nDaily sales dataset columns:")
print(daily_sales_final_df.columns.tolist())

print("\nDaily sales records:", len(daily_sales_final_df))
print("Daily sales total units:", daily_sales_final_df["Units_Sold"].sum())

In [ ]:
# Load the six saved datasets

products_df = pd.read_csv(raw_data_dir / "Products.csv")

suppliers_df = pd.read_csv(raw_data_dir / "Suppliers.csv")

product_supplier_df = pd.read_csv(
    raw_data_dir / "Product_Supplier_Mapping.csv"
)

daily_sales_df = pd.read_csv(
    raw_data_dir / "Daily_Sales.csv",
    parse_dates=["Date"]
)

purchase_orders_df = pd.read_csv(
    raw_data_dir / "Purchase_Orders.csv",
    parse_dates=[
        "PO_Date",
        "Expected_Delivery_Date",
        "First_Receipt_Date",
        "Last_Receipt_Date"
    ]
)

purchase_order_receipts_df = pd.read_csv(
    raw_data_dir / "Purchase_Order_Receipts.csv",
    parse_dates=["Receipt_Date"]
)

# Review dataset coverage
print("DATASET RECORD COUNTS")
print("----------------------")
print("Products:", len(products_df))
print("Suppliers:", len(suppliers_df))
print("Product-Supplier Mapping:", len(product_supplier_df))
print("Daily Sales:", len(daily_sales_df))
print("Purchase Orders:", len(purchase_orders_df))
print("Purchase Order Receipts:", len(purchase_order_receipts_df))

print("\nDAILY SALES COVERAGE")
print("--------------------")
print("Start date:", daily_sales_df["Date"].min())
print("End date:", daily_sales_df["Date"].max())
print("Unique SKUs:", daily_sales_df["SKU_ID"].nunique())
print("Categories:", daily_sales_df["Category"].nunique())
print("Total units sold:", daily_sales_df["Units_Sold"].sum())
print("In-stock SKU-days:", daily_sales_df["In_Stock_Flag"].sum())
print("Out-of-stock SKU-days:", (~daily_sales_df["In_Stock_Flag"]).sum())

In [ ]:
# Category-level sales summary

category_sales_df = (
    daily_sales_df
    .groupby(["Year", "Category"], as_index=False)
    .agg(
        Total_Units_Sold=("Units_Sold", "sum"),
        In_Stock_SKU_Days=("In_Stock_Flag", "sum"),
        Total_SKU_Days=("SKU_ID", "count"),
        Unique_SKUs=("SKU_ID", "nunique")
    )
)

# Calculate in-stock percentage
category_sales_df["In_Stock_Percentage"] = (
    category_sales_df["In_Stock_SKU_Days"]
    / category_sales_df["Total_SKU_Days"] * 100
).round(2)

# Calculate sales share within each year
category_sales_df["Sales_Share_Percentage"] = (
    category_sales_df["Total_Units_Sold"]
    / category_sales_df.groupby("Year")["Total_Units_Sold"].transform("sum")
    * 100
).round(2)

# Display results
display(
    category_sales_df.sort_values(
        ["Year", "Total_Units_Sold"],
        ascending=[True, False]
    )
)

In [ ]:
# Compare category sales between 2024 and 2025

annual_category_sales = (
    daily_sales_df
    .pivot_table(
        index="Category",
        columns="Year",
        values="Units_Sold",
        aggfunc="sum"
    )
    .reset_index()
)

annual_category_sales.columns.name = None

annual_category_sales = annual_category_sales.rename(
    columns={
        2024: "Sales_2024",
        2025: "Sales_2025"
    }
)

annual_category_sales["Change_in_Units"] = (
    annual_category_sales["Sales_2025"]
    - annual_category_sales["Sales_2024"]
)

annual_category_sales["Change_Percentage"] = (
    annual_category_sales["Change_in_Units"]
    / annual_category_sales["Sales_2024"] * 100
).round(2)

display(
    annual_category_sales.sort_values(
        "Sales_2024",
        ascending=False
    )
)

In [ ]:
# Monthly sales by category and year

daily_sales_df["Month"] = daily_sales_df["Date"].dt.month

monthly_sales_df = (
    daily_sales_df
    .groupby(["Year", "Month", "Category"], as_index=False)
    .agg(
        Total_Units_Sold=("Units_Sold", "sum")
    )
)

# Overall monthly sales
monthly_total_sales_df = (
    monthly_sales_df
    .groupby(["Year", "Month"], as_index=False)
    ["Total_Units_Sold"]
    .sum()
)

# Compare the monthly pattern across years
monthly_sales_pivot = monthly_total_sales_df.pivot(
    index="Month",
    columns="Year",
    values="Total_Units_Sold"
).reset_index()

monthly_sales_pivot.columns.name = None

monthly_sales_pivot = monthly_sales_pivot.rename(
    columns={
        2024: "Sales_2024",
        2025: "Sales_2025"
    }
)

monthly_sales_pivot["Average_Monthly_Sales"] = (
    monthly_sales_pivot[["Sales_2024", "Sales_2025"]]
    .mean(axis=1)
).round(0)

display(monthly_sales_pivot)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    monthly_sales_pivot["Month"],
    monthly_sales_pivot["Sales_2024"],
    marker="o",
    label="2024"
)

plt.plot(
    monthly_sales_pivot["Month"],
    monthly_sales_pivot["Sales_2025"],
    marker="o",
    label="2025"
)

plt.title("Monthly Sales Trend – 2024 vs 2025")
plt.xlabel("Month")
plt.ylabel("Units Sold")
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Add demand pattern to daily sales data
daily_sales_df = daily_sales_df.merge(
    products_df[["SKU_ID", "Demand_Pattern"]],
    on="SKU_ID",
    how="left"
)

# Calculate SKU-level sales and availability metrics
sku_demand_summary = (
    daily_sales_df
    .groupby(["SKU_ID", "Category", "Demand_Pattern"], as_index=False)
    .agg(
        Total_Units_Sold=("Units_Sold", "sum"),
        In_Stock_Days=("In_Stock_Flag", "sum"),
        Total_Days=("Date", "count"),
        Out_of_Stock_Days=(
            "In_Stock_Flag",
            lambda x: (~x).sum()
        )
    )
)

# Calculate average daily sales using in-stock days
sku_demand_summary["Avg_Daily_Sales_In_Stock"] = (
    sku_demand_summary["Total_Units_Sold"]
    / sku_demand_summary["In_Stock_Days"].replace(0, np.nan)
).round(2)

# Summarise by demand pattern
pattern_summary = (
    sku_demand_summary
    .groupby("Demand_Pattern", as_index=False)
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Total_Units_Sold=("Total_Units_Sold", "sum"),
        Avg_Daily_Sales=("Avg_Daily_Sales_In_Stock", "mean"),
        Avg_In_Stock_Days=("In_Stock_Days", "mean"),
        Avg_Out_of_Stock_Days=("Out_of_Stock_Days", "mean")
    )
)

display(
    pattern_summary.sort_values(
        "Total_Units_Sold",
        ascending=False
    )
)

In [ ]:
# Summarise actual sales and availability by year and demand pattern

pattern_year_summary = (
    daily_sales_df
    .groupby(["Year", "Demand_Pattern"], as_index=False)
    .agg(
        Total_Units_Sold=("Units_Sold", "sum"),
        In_Stock_SKU_Days=("In_Stock_Flag", "sum"),
        Total_SKU_Days=("SKU_ID", "count")
    )
)

# Calculate average sales per in-stock SKU-day
pattern_year_summary["Avg_Daily_Sales_In_Stock"] = (
    pattern_year_summary["Total_Units_Sold"]
    / pattern_year_summary["In_Stock_SKU_Days"]
).round(2)

# Calculate stockout percentage
pattern_year_summary["Out_of_Stock_Percentage"] = (
    100 -
    pattern_year_summary["In_Stock_SKU_Days"]
    / pattern_year_summary["Total_SKU_Days"] * 100
).round(2)

display(
    pattern_year_summary.sort_values(
        ["Demand_Pattern", "Year"]
    )
)

In [ ]:
# Monthly sales by demand pattern

monthly_pattern_df = (
    daily_sales_df
    .groupby(
        ["Month", "Demand_Pattern"],
        as_index=False
    )
    .agg(
        Total_Units_Sold=("Units_Sold", "sum"),
        In_Stock_SKU_Days=("In_Stock_Flag", "sum")
    )
)

# Calculate average sales per in-stock SKU-day
monthly_pattern_df["Avg_Daily_Sales_In_Stock"] = (
    monthly_pattern_df["Total_Units_Sold"]
    / monthly_pattern_df["In_Stock_SKU_Days"].replace(0, np.nan)
).round(2)

# Compare monthly average daily sales across demand patterns
monthly_pattern_pivot = monthly_pattern_df.pivot(
    index="Month",
    columns="Demand_Pattern",
    values="Avg_Daily_Sales_In_Stock"
).reset_index()

monthly_pattern_pivot.columns.name = None

display(monthly_pattern_pivot)

In [ ]:
# SKU-level demand behaviour indicators

sku_demand_indicators = (
    daily_sales_df
    .groupby("SKU_ID")
    .apply(
        lambda x: pd.Series({
            "Total_Units_Sold": x["Units_Sold"].sum(),
            "In_Stock_Days": x["In_Stock_Flag"].sum(),
            "Zero_Sales_Days": (
                (x["In_Stock_Flag"]) &
                (x["Units_Sold"] == 0)
            ).sum(),
            "Average_Daily_Demand": (
                x.loc[x["In_Stock_Flag"], "Units_Sold"].mean()
            ),
            "Demand_Std_Dev": (
                x.loc[x["In_Stock_Flag"], "Units_Sold"].std()
            )
        })
    )
    .reset_index()
)

sku_demand_indicators["Zero_Sales_Pct"] = (
    sku_demand_indicators["Zero_Sales_Days"]
    / sku_demand_indicators["In_Stock_Days"]
    * 100
).round(2)

sku_demand_indicators["Demand_CV"] = (
    sku_demand_indicators["Demand_Std_Dev"]
    / sku_demand_indicators["Average_Daily_Demand"]
).round(2)

display(sku_demand_indicators.head())

In [ ]:
# Calculate intermittency indicators

sku_demand_indicators["Non_Zero_Sales_Days"] = (
    sku_demand_indicators["In_Stock_Days"]
    - sku_demand_indicators["Zero_Sales_Days"]
)

sku_demand_indicators["ADI"] = (
    sku_demand_indicators["In_Stock_Days"]
    / sku_demand_indicators["Non_Zero_Sales_Days"].replace(0, np.nan)
).round(2)

sku_demand_indicators["Demand_CV_Squared"] = (
    sku_demand_indicators["Demand_CV"] ** 2
).round(2)

display(
    sku_demand_indicators[
        [
            "SKU_ID",
            "Zero_Sales_Pct",
            "ADI",
            "Demand_CV",
            "Demand_CV_Squared"
        ]
    ].head()
)

print("\nIndicator distribution:")
display(
    sku_demand_indicators[
        ["Zero_Sales_Pct", "ADI", "Demand_CV", "Demand_CV_Squared"]
    ].describe().round(2)
)

In [ ]:
# Calculate demand variability using only days with positive sales

positive_demand_cv = (
    daily_sales_df[
        daily_sales_df["In_Stock_Flag"] &
        (daily_sales_df["Units_Sold"] > 0)
    ]
    .groupby("SKU_ID")["Units_Sold"]
    .agg(["mean", "std"])
    .reset_index()
)

positive_demand_cv["Positive_Demand_CV"] = (
    positive_demand_cv["std"] / positive_demand_cv["mean"]
)

positive_demand_cv["Positive_Demand_CV_Squared"] = (
    positive_demand_cv["Positive_Demand_CV"] ** 2
)

sku_demand_indicators = sku_demand_indicators.drop(
    columns=["Demand_CV", "Demand_CV_Squared"],
    errors="ignore"
)

sku_demand_indicators = sku_demand_indicators.merge(
    positive_demand_cv[
        ["SKU_ID", "Positive_Demand_CV", "Positive_Demand_CV_Squared"]
    ],
    on="SKU_ID",
    how="left"
)

display(
    sku_demand_indicators[
        [
            "SKU_ID",
            "ADI",
            "Positive_Demand_CV",
            "Positive_Demand_CV_Squared"
        ]
    ].head()
)

display(
    sku_demand_indicators[
        ["ADI", "Positive_Demand_CV_Squared"]
    ].describe().round(2)
)

In [ ]:
# Classify SKU demand variability using ADI and CV²

conditions = [
    (
        (sku_demand_indicators["ADI"] <= 1.32) &
        (sku_demand_indicators["Positive_Demand_CV_Squared"] <= 0.49)
    ),
    (
        (sku_demand_indicators["ADI"] <= 1.32) &
        (sku_demand_indicators["Positive_Demand_CV_Squared"] > 0.49)
    ),
    (
        (sku_demand_indicators["ADI"] > 1.32) &
        (sku_demand_indicators["Positive_Demand_CV_Squared"] <= 0.49)
    ),
    (
        (sku_demand_indicators["ADI"] > 1.32) &
        (sku_demand_indicators["Positive_Demand_CV_Squared"] > 0.49)
    )
]

demand_types = [
    "Smooth",
    "Erratic",
    "Intermittent",
    "Lumpy"
]

sku_demand_indicators["Demand_Variability_Type"] = np.select(
    conditions,
    demand_types,
    default="Unclassified"
)

demand_type_summary = (
    sku_demand_indicators
    .groupby("Demand_Variability_Type")
    .agg(
        SKU_Count=("SKU_ID", "count"),
        Avg_ADI=("ADI", "mean"),
        Avg_CV_Squared=("Positive_Demand_CV_Squared", "mean")
    )
    .reset_index()
)

demand_type_summary["SKU_Percentage"] = (
    demand_type_summary["SKU_Count"] /
    len(sku_demand_indicators) * 100
).round(2)

display(demand_type_summary.round(2))

In [ ]:
# Monthly SKU sales summary

monthly_sku_sales = (
    daily_sales_df
    .groupby(
        ["SKU_ID", daily_sales_df["Date"].dt.to_period("M")],
        as_index=False
    )
    .agg(
        Total_Units_Sold=("Units_Sold", "sum"),
        In_Stock_Days=("In_Stock_Flag", "sum")
    )
)

monthly_sku_sales = monthly_sku_sales.rename(
    columns={"Date": "Month"}
)

monthly_sku_sales["Month"] = (
    monthly_sku_sales["Month"].astype(str)
)

display(monthly_sku_sales.head())
print("Rows:", len(monthly_sku_sales))

In [ ]:
# Calculate average daily sales on in-stock days by SKU and year

monthly_sku_sales["Year"] = (
    monthly_sku_sales["Month"].str[:4].astype(int)
)

annual_sku_demand = (
    monthly_sku_sales
    .groupby(["SKU_ID", "Year"], as_index=False)
    .agg(
        Total_Units_Sold=("Total_Units_Sold", "sum"),
        In_Stock_Days=("In_Stock_Days", "sum")
    )
)

annual_sku_demand["Avg_Daily_Demand"] = (
    annual_sku_demand["Total_Units_Sold"]
    / annual_sku_demand["In_Stock_Days"].replace(0, np.nan)
)

annual_demand_pivot = annual_sku_demand.pivot(
    index="SKU_ID",
    columns="Year",
    values="Avg_Daily_Demand"
).reset_index()

annual_demand_pivot = annual_demand_pivot.rename(
    columns={
        2024: "Avg_Daily_Demand_2024",
        2025: "Avg_Daily_Demand_2025"
    }
)

annual_demand_pivot["Demand_Change_Pct"] = (
    (
        annual_demand_pivot["Avg_Daily_Demand_2025"]
        / annual_demand_pivot["Avg_Daily_Demand_2024"]
    ) - 1
) * 100

display(annual_demand_pivot.head().round(2))

print("SKUs available for comparison:",
      annual_demand_pivot["Demand_Change_Pct"].notna().sum())

In [ ]:
# Distribution of year-on-year demand changes

print("Demand change distribution:")
display(
    annual_demand_pivot["Demand_Change_Pct"]
    .describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
    .round(2)
)

# Count SKUs across different change ranges

change_ranges = pd.cut(
    annual_demand_pivot["Demand_Change_Pct"],
    bins=[-np.inf, -30, -20, -10, 10, 20, 30, np.inf],
    labels=[
        "Below -30%",
        "-30% to -20%",
        "-20% to -10%",
        "-10% to +10%",
        "+10% to +20%",
        "+20% to +30%",
        "Above +30%"
    ]
)

change_summary = (
    change_ranges
    .value_counts(sort=False)
    .rename_axis("Demand_Change_Range")
    .reset_index(name="SKU_Count")
)

change_summary["SKU_Percentage"] = (
    change_summary["SKU_Count"] / len(annual_demand_pivot) * 100
).round(2)

display(change_summary)

In [ ]:
# Classify preliminary demand trends

annual_demand_pivot["Trend_Classification"] = np.select(
    [
        annual_demand_pivot["Demand_Change_Pct"] > 10,
        annual_demand_pivot["Demand_Change_Pct"] < -10
    ],
    [
        "Growing",
        "Declining"
    ],
    default="Stable"
)

trend_summary = (
    annual_demand_pivot
    .groupby("Trend_Classification")
    .agg(
        SKU_Count=("SKU_ID", "count"),
        Avg_Demand_Change=("Demand_Change_Pct", "mean")
    )
    .reset_index()
)

trend_summary["SKU_Percentage"] = (
    trend_summary["SKU_Count"] /
    len(annual_demand_pivot) * 100
).round(2)

display(trend_summary.round(2))

In [ ]:
# Calculate monthly average daily demand by SKU and year

monthly_seasonality = (
    daily_sales_df
    .groupby(
        [
            "SKU_ID",
            daily_sales_df["Date"].dt.year.rename("Year"),
            daily_sales_df["Date"].dt.month.rename("Month")
        ]
    )
    .apply(
        lambda x: pd.Series({
            "Total_Units_Sold": x["Units_Sold"].sum(),
            "In_Stock_Days": x["In_Stock_Flag"].sum()
        }),
        include_groups=False
    )
    .reset_index()
)

monthly_seasonality["Avg_Daily_Demand"] = (
    monthly_seasonality["Total_Units_Sold"]
    / monthly_seasonality["In_Stock_Days"].replace(0, np.nan)
)

display(monthly_seasonality.head())
print("Rows:", len(monthly_seasonality))

In [ ]:
# Compare normalised monthly demand profiles across 2024 and 2025

profile_2024 = (
    monthly_seasonality[monthly_seasonality["Year"] == 2024]
    .pivot(index="SKU_ID", columns="Month", values="Avg_Daily_Demand")
    .reindex(columns=range(1, 13))
)

profile_2025 = (
    monthly_seasonality[monthly_seasonality["Year"] == 2025]
    .pivot(index="SKU_ID", columns="Month", values="Avg_Daily_Demand")
    .reindex(columns=range(1, 13))
)

# Normalise each year's monthly demand by that SKU's annual monthly average

normalised_2024 = profile_2024.div(
    profile_2024.mean(axis=1), axis=0
)

normalised_2025 = profile_2025.div(
    profile_2025.mean(axis=1), axis=0
)

# Calculate correlation between the two annual monthly profiles

profile_values = np.concatenate(
    [
        normalised_2024.to_numpy(),
        normalised_2025.to_numpy()
    ],
    axis=1
)

def calculate_profile_correlation(row):
    first_year = row[:12]
    second_year = row[12:]

    valid = np.isfinite(first_year) & np.isfinite(second_year)

    if valid.sum() < 3:
        return np.nan

    return np.corrcoef(
        first_year[valid],
        second_year[valid]
    )[0, 1]

seasonality_indicators = pd.DataFrame({
    "SKU_ID": normalised_2024.index,
    "Seasonal_Profile_Correlation": [
        calculate_profile_correlation(row)
        for row in profile_values
    ]
})

# Calculate variation in the average normalised monthly profile

average_seasonal_profile = (
    normalised_2024 + normalised_2025
) / 2

seasonality_indicators["Seasonal_Index_CV"] = (
    average_seasonal_profile.std(axis=1, ddof=0)
    / average_seasonal_profile.mean(axis=1)
).values

display(seasonality_indicators.head().round(3))

print("\nSeasonality indicator distribution:")
display(
    seasonality_indicators[
        ["Seasonal_Profile_Correlation", "Seasonal_Index_CV"]
    ].describe().round(3)
)

In [ ]:
# Compare normalised monthly demand profiles across 2024 and 2025

profile_2024 = (
    monthly_seasonality[monthly_seasonality["Year"] == 2024]
    .pivot(index="SKU_ID", columns="Month", values="Avg_Daily_Demand")
    .reindex(columns=range(1, 13))
)

profile_2025 = (
    monthly_seasonality[monthly_seasonality["Year"] == 2025]
    .pivot(index="SKU_ID", columns="Month", values="Avg_Daily_Demand")
    .reindex(columns=range(1, 13))
)

# Normalise each year's monthly demand by that SKU's annual monthly average

normalised_2024 = profile_2024.div(
    profile_2024.mean(axis=1), axis=0
)

normalised_2025 = profile_2025.div(
    profile_2025.mean(axis=1), axis=0
)

# Calculate correlation between the two annual monthly profiles

profile_values = np.concatenate(
    [
        normalised_2024.to_numpy(),
        normalised_2025.to_numpy()
    ],
    axis=1
)

def calculate_profile_correlation(row):
    first_year = row[:12]
    second_year = row[12:]

    valid = np.isfinite(first_year) & np.isfinite(second_year)

    if valid.sum() < 3:
        return np.nan

    return np.corrcoef(
        first_year[valid],
        second_year[valid]
    )[0, 1]

seasonality_indicators = pd.DataFrame({
    "SKU_ID": normalised_2024.index,
    "Seasonal_Profile_Correlation": [
        calculate_profile_correlation(row)
        for row in profile_values
    ]
})

# Calculate variation in the average normalised monthly profile

average_seasonal_profile = (
    normalised_2024 + normalised_2025
) / 2

seasonality_indicators["Seasonal_Index_CV"] = (
    average_seasonal_profile.std(axis=1, ddof=0)
    / average_seasonal_profile.mean(axis=1)
).values

display(seasonality_indicators.head().round(3))

print("\nSeasonality indicator distribution:")
display(
    seasonality_indicators[
        ["Seasonal_Profile_Correlation", "Seasonal_Index_CV"]
    ].describe().round(3)
)

In [ ]:
# Classify seasonal behaviour

seasonality_indicators["Seasonality_Classification"] = np.where(
    (
        (seasonality_indicators["Seasonal_Profile_Correlation"] >= 0.70) &
        (seasonality_indicators["Seasonal_Index_CV"] >= 0.20)
    ),
    "Seasonal",
    "Non-Seasonal"
)

seasonality_summary = (
    seasonality_indicators
    .groupby("Seasonality_Classification")
    .agg(
        SKU_Count=("SKU_ID", "count"),
        Avg_Profile_Correlation=("Seasonal_Profile_Correlation", "mean"),
        Avg_Seasonal_CV=("Seasonal_Index_CV", "mean")
    )
    .reset_index()
)

seasonality_summary["SKU_Percentage"] = (
    seasonality_summary["SKU_Count"] /
    len(seasonality_indicators) * 100
).round(2)

display(seasonality_summary.round(3))

In [ ]:
# Combine independently derived SKU demand classifications

sku_demand_profile = (
    sku_demand_indicators[
        [
            "SKU_ID",
            "ADI",
            "Positive_Demand_CV_Squared",
            "Demand_Variability_Type",
            "Zero_Sales_Pct",
            "Average_Daily_Demand"
        ]
    ]
    .merge(
        annual_demand_pivot[
            ["SKU_ID", "Demand_Change_Pct", "Trend_Classification"]
        ],
        on="SKU_ID",
        how="left"
    )
    .merge(
        seasonality_indicators[
            [
                "SKU_ID",
                "Seasonal_Profile_Correlation",
                "Seasonal_Index_CV",
                "Seasonality_Classification"
            ]
        ],
        on="SKU_ID",
        how="left"
    )
)

print("Total SKUs:", len(sku_demand_profile))
print("Unique SKUs:", sku_demand_profile["SKU_ID"].nunique())
print("Missing values:")
display(sku_demand_profile.isna().sum())

display(sku_demand_profile.head(10).round(3))

In [ ]:
# Examine combinations of demand characteristics

print("Demand variability vs seasonality:")
display(
    pd.crosstab(
        sku_demand_profile["Demand_Variability_Type"],
        sku_demand_profile["Seasonality_Classification"],
        margins=True
    )
)

print("\nDemand trend vs seasonality:")
display(
    pd.crosstab(
        sku_demand_profile["Trend_Classification"],
        sku_demand_profile["Seasonality_Classification"],
        margins=True
    )
)

print("\nDemand variability vs trend:")
display(
    pd.crosstab(
        sku_demand_profile["Demand_Variability_Type"],
        sku_demand_profile["Trend_Classification"],
        margins=True
    )
)

In [ ]:

# Define historical evaluation periods

evaluation_periods = {
    "Peak_2024": {
        "train_end": "2024-09-30",
        "test_start": "2024-10-01",
        "test_end": "2024-12-29"
    },
    "Peak_2025": {
        "train_end": "2025-09-30",
        "test_start": "2025-10-01",
        "test_end": "2025-12-29"
    }
}

for period, dates in evaluation_periods.items():
    print(f"\n{period}")
    print("Training ends:", dates["train_end"])
    print("Testing:", dates["test_start"], "to", dates["test_end"])


In [ ]:

# Check evaluation-period coverage

for period, dates in evaluation_periods.items():
    start_date = pd.Timestamp(dates["test_start"])
    end_date = pd.Timestamp(dates["test_end"])

    period_df = daily_sales_df[
        daily_sales_df["Date"].between(start_date, end_date)
    ]

    print(f"\n{period}")
    print("Calendar days:", (end_date - start_date).days + 1)
    print("Rows:", len(period_df))
    print("Unique SKUs:", period_df["SKU_ID"].nunique())
    print("In-stock SKU-days:", period_df["In_Stock_Flag"].sum())
    print("Out-of-stock SKU-days:", (~period_df["In_Stock_Flag"]).sum())


In [ ]:

# Calculate stock availability by SKU for each evaluation period

for period, dates in evaluation_periods.items():
    start_date = pd.Timestamp(dates["test_start"])
    end_date = pd.Timestamp(dates["test_end"])

    period_df = daily_sales_df[
        daily_sales_df["Date"].between(start_date, end_date)
    ]

    availability = (
        period_df.groupby("SKU_ID")
        .agg(
            In_Stock_Days=("In_Stock_Flag", "sum"),
            Total_Days=("Date", "count")
        )
    )

    availability["Availability_Pct"] = (
        availability["In_Stock_Days"] / availability["Total_Days"] * 100
    )

    print(f"\n{period}")
    print("SKUs with 90 in-stock days:",
          (availability["In_Stock_Days"] == 90).sum())
    print("SKUs with at least 75 in-stock days:",
          (availability["In_Stock_Days"] >= 75).sum())
    print("SKUs with fewer than 45 in-stock days:",
          (availability["In_Stock_Days"] < 45).sum())
    print("Average availability:",
          round(availability["Availability_Pct"].mean(), 2), "%")


In [ ]:

# Combine demand profiles with stock availability in each evaluation period

for period, dates in evaluation_periods.items():
    start_date = pd.Timestamp(dates["test_start"])
    end_date = pd.Timestamp(dates["test_end"])

    period_df = daily_sales_df[
        daily_sales_df["Date"].between(start_date, end_date)
    ]

    sku_availability = (
        period_df.groupby("SKU_ID")
        .agg(
            In_Stock_Days=("In_Stock_Flag", "sum"),
            Total_Days=("Date", "count")
        )
        .reset_index()
    )

    sku_availability["Availability_Pct"] = (
        sku_availability["In_Stock_Days"]
        / sku_availability["Total_Days"] * 100
    )

    profile_availability = sku_availability.merge(
        sku_demand_profile[
            ["SKU_ID", "Demand_Variability_Type",
             "Trend_Classification", "Seasonality_Classification"]
        ],
        on="SKU_ID",
        how="left"
    )

    print(f"\n{period} - Availability by demand variability")
    display(
        profile_availability.groupby("Demand_Variability_Type")
        ["Availability_Pct"].agg(["count", "mean", "median"]).round(2)
    )

    print(f"\n{period} - Availability by seasonality")
    display(
        profile_availability.groupby("Seasonality_Classification")
        ["Availability_Pct"].agg(["count", "mean", "median"]).round(2)
    )


In [ ]:

# Prepare daily time series for one SKU

sample_sku = "SKU-0001"

sample_sku_df = (
    daily_sales_df[
        daily_sales_df["SKU_ID"] == sample_sku
    ]
    .sort_values("Date")
    .copy()
    .set_index("Date")
)

# Keep sales and availability as separate fields
sample_sku_ts = sample_sku_df[
    ["Units_Sold", "In_Stock_Flag"]
].copy()

print("SKU:", sample_sku)
print("Date range:", sample_sku_ts.index.min(), "to",
      sample_sku_ts.index.max())
print("Total calendar days:", len(sample_sku_ts))
print("In-stock days:", sample_sku_ts["In_Stock_Flag"].sum())
print("Out-of-stock days:", (~sample_sku_ts["In_Stock_Flag"]).sum())

print("\nFirst 10 records:")
display(sample_sku_ts.head(10))


In [ ]:

# Find a SKU with stockouts during the historical period

sku_stockout_counts = (
    daily_sales_df.groupby("SKU_ID")
    .agg(
        Out_of_Stock_Days=(
            "In_Stock_Flag",
            lambda x: (~x).sum()
        )
    )
    .sort_values("Out_of_Stock_Days", ascending=False)
)

stockout_sku = sku_stockout_counts.index[0]

stockout_sku_df = (
    daily_sales_df[
        daily_sales_df["SKU_ID"] == stockout_sku
    ]
    .sort_values("Date")
    .copy()
    .set_index("Date")
)

stockout_sku_ts = stockout_sku_df[
    ["Units_Sold", "In_Stock_Flag"]
].copy()

print("Selected SKU:", stockout_sku)
print("Total calendar days:", len(stockout_sku_ts))
print("In-stock days:", stockout_sku_ts["In_Stock_Flag"].sum())
print("Out-of-stock days:", (~stockout_sku_ts["In_Stock_Flag"]).sum())

print("\nFirst 20 records:")
display(stockout_sku_ts.head(20))


In [ ]:

# Compare daily sales rates for the selected SKU

total_units_sold = stockout_sku_ts["Units_Sold"].sum()

in_stock_days = stockout_sku_ts["In_Stock_Flag"].sum()
total_days = len(stockout_sku_ts)

average_daily_sales_all_days = total_units_sold / total_days
average_daily_sales_in_stock_days = total_units_sold / in_stock_days

print("Selected SKU:", stockout_sku)
print("Total units sold:", total_units_sold)
print("Total calendar days:", total_days)
print("In-stock days:", in_stock_days)
print("Out-of-stock days:", total_days - in_stock_days)

print("\nAverage daily sales across all calendar days:",
      round(average_daily_sales_all_days, 3))

print("Average daily sales on in-stock days:",
      round(average_daily_sales_in_stock_days, 3))


In [ ]:

# Create an availability-aware sales series

stockout_sku_ts["Sales_For_Model"] = (
    stockout_sku_ts["Units_Sold"]
    .where(stockout_sku_ts["In_Stock_Flag"], np.nan)
)

print("Original calendar days:", len(stockout_sku_ts))
print("Missing observations:",
      stockout_sku_ts["Sales_For_Model"].isna().sum())
print("Genuine zero-sales days:",
      (stockout_sku_ts["Sales_For_Model"] == 0).sum())
print("Available sales observations:",
      stockout_sku_ts["Sales_For_Model"].notna().sum())

print("\nSample around the first stockout:")
display(stockout_sku_ts.loc["2024-01-10":"2024-01-22"])


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(
    stockout_sku_ts.index,
    stockout_sku_ts["Sales_For_Model"],
    linewidth=0.8,
    label="Daily sales (in-stock days)"
)

ax.set_title(f"Daily Sales and Availability – {stockout_sku}")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# First Moving Average baseline for SKU-0186

moving_average_window = 28

for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])
    history_start = test_start - pd.Timedelta(
        days=moving_average_window
    )

    history = stockout_sku_ts.loc[
        history_start:test_start - pd.Timedelta(days=1)
    ]

    available_sales = history.loc[
        history["In_Stock_Flag"], "Units_Sold"
    ]

    forecast_value = available_sales.mean()

    print(f"\n{period}")
    print("Historical window:", history_start.date(),
          "to", (test_start - pd.Timedelta(days=1)).date())
    print("Calendar days:", len(history))
    print("Available observations:", len(available_sales))
    print("Average daily sales:", round(forecast_value, 3))
    print("90-day forecast:", round(forecast_value * 90, 1))


In [ ]:

# Evaluate the 28-day Moving Average for SKU-0186

moving_average_results = []

for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])
    history_start = test_start - pd.Timedelta(days=28)

    history = stockout_sku_ts.loc[
        history_start:test_start - pd.Timedelta(days=1)
    ]

    available_history = history.loc[
        history["In_Stock_Flag"], "Units_Sold"
    ]

    forecast_value = available_history.mean()

    test_data = stockout_sku_ts.loc[test_start:test_end].copy()

    # Evaluate only days when the SKU was available
    actual = test_data.loc[
        test_data["In_Stock_Flag"], "Units_Sold"
    ]

    forecast = pd.Series(forecast_value, index=actual.index)

    errors = forecast - actual

    mae = errors.abs().mean()
    wape = errors.abs().sum() / actual.sum() * 100
    bias = errors.sum() / actual.sum() * 100

    moving_average_results.append({
        "Evaluation_Period": period,
        "Forecast_Daily_Avg": forecast_value,
        "Evaluated_Days": len(actual),
        "Actual_Units": actual.sum(),
        "Forecast_Units": forecast.sum(),
        "MAE": mae,
        "WAPE_Pct": wape,
        "Bias_Pct": bias
    })

moving_average_evaluation = pd.DataFrame(moving_average_results)

display(moving_average_evaluation.round(2))


In [ ]:

# Inspect the final 14 days before each evaluation period

for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])
    history_start = test_start - pd.Timedelta(days=14)

    recent_history = stockout_sku_ts.loc[
        history_start:test_start - pd.Timedelta(days=1),
        ["Units_Sold", "In_Stock_Flag"]
    ]

    print(f"\n{period}")
    display(recent_history)


In [ ]:

# Weekly Seasonal Naive forecast for SKU-0186

seasonal_naive_results = []

for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])

    # Use the 28 days before the evaluation period
    history_start = test_start - pd.Timedelta(days=28)

    history = stockout_sku_ts.loc[
        history_start:test_start - pd.Timedelta(days=1)
    ].copy()

    # Exclude unavailable days
    available_history = history.loc[
        history["In_Stock_Flag"]
    ].copy()

    # Find the latest available sales observation for each weekday
    weekday_pattern = (
        available_history
        .groupby(available_history.index.dayofweek)["Units_Sold"]
        .last()
    )

    # Generate the complete 90-day forecast
    forecast_dates = pd.date_range(
        start=test_start,
        end=test_end,
        freq="D"
    )

    forecast = pd.Series(
        forecast_dates.dayofweek.map(weekday_pattern),
        index=forecast_dates
    )

    print(f"\n{period}")
    print("Historical observations:", len(available_history))
    print("Weekdays represented:", len(weekday_pattern))
    print("\nWeekly forecast pattern:")
    display(weekday_pattern.rename_axis("Weekday").to_frame("Forecast_Units"))

    print("Average daily forecast:", round(forecast.mean(), 3))
    print("90-day forecast:", round(forecast.sum(), 1))


In [ ]:

# Evaluate Weekly Seasonal Naive for SKU-0186

seasonal_naive_evaluation_results = []

for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])
    history_start = test_start - pd.Timedelta(days=28)

    history = stockout_sku_ts.loc[
        history_start:test_start - pd.Timedelta(days=1)
    ]

    available_history = history.loc[
        history["In_Stock_Flag"]
    ]

    # Latest available observation for each weekday
    weekday_pattern = (
        available_history
        .groupby(available_history.index.dayofweek)["Units_Sold"]
        .last()
    )

    # Generate predictions for all 90 calendar days
    forecast_dates = pd.date_range(
        start=test_start,
        end=test_end,
        freq="D"
    )

    forecast = pd.Series(
        forecast_dates.dayofweek.map(weekday_pattern).to_numpy(),
        index=forecast_dates
    )

    test_data = stockout_sku_ts.loc[test_start:test_end]

    # Evaluate only in-stock days
    actual = test_data.loc[
        test_data["In_Stock_Flag"], "Units_Sold"
    ]

    aligned_forecast = forecast.loc[actual.index]

    errors = aligned_forecast - actual

    seasonal_naive_evaluation_results.append({
        "Evaluation_Period": period,
        "Evaluated_Days": len(actual),
        "Actual_Units": actual.sum(),
        "Forecast_Units": aligned_forecast.sum(),
        "MAE": errors.abs().mean(),
        "WAPE_Pct": errors.abs().sum() / actual.sum() * 100,
        "Bias_Pct": errors.sum() / actual.sum() * 100
    })

seasonal_naive_evaluation = pd.DataFrame(
    seasonal_naive_evaluation_results
)

display(seasonal_naive_evaluation.round(2))


In [ ]:

try:
    import statsmodels
    print("Statsmodels version:", statsmodels.__version__)
    print("Library is available.")
except ImportError:
    print("Statsmodels is not installed.")


In [ ]:

from statsmodels.tsa.holtwinters import SimpleExpSmoothing

for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])

    history = stockout_sku_ts.loc[
        stockout_sku_ts.index < test_start,
        "Sales_For_Model"
    ]

    model = SimpleExpSmoothing(
        history,
        initialization_method="estimated"
    )

    fitted_model = model.fit(optimized=True)

    print(f"\n{period}")
    print("Training observations:", history.notna().sum())
    print(
        "Estimated smoothing level (alpha):",
        round(fitted_model.params["smoothing_level"], 4)
    )
    print(
        "Last estimated demand level:",
        round(fitted_model.level.iloc[-1], 3)
    )


In [ ]:

from scipy.optimize import minimize_scalar
import numpy as np

def fit_missing_aware_ses(series):
    values = series.to_numpy(dtype=float)
    valid_positions = np.flatnonzero(~np.isnan(values))

    if len(valid_positions) < 2:
        raise ValueError("Insufficient observations.")

    def calculate_sse(alpha):
        level = values[valid_positions[0]]
        errors = []

        for value in values[valid_positions[1:]]:
            errors.append(value - level)
            level = alpha * value + (1 - alpha) * level

        return np.sum(np.square(errors))

    result = minimize_scalar(
        calculate_sse,
        bounds=(0.0001, 0.9999),
        method="bounded"
    )

    alpha = result.x
    level = values[valid_positions[0]]

    for value in values[valid_positions[1:]]:
        level = alpha * value + (1 - alpha) * level

    return alpha, level, result.fun


for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])

    history = stockout_sku_ts.loc[
        stockout_sku_ts.index < test_start,
        "Sales_For_Model"
    ]

    alpha, final_level, sse = fit_missing_aware_ses(history)

    print(f"\n{period}")
    print("Training observations:", history.notna().sum())
    print("Estimated alpha:", round(alpha, 4))
    print("Final estimated demand level:", round(final_level, 3))


In [ ]:

ses_forecasts = {}

for period, dates in evaluation_periods.items():
    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])

    history = stockout_sku_ts.loc[
        stockout_sku_ts.index < test_start,
        "Sales_For_Model"
    ]

    alpha, final_level, sse = fit_missing_aware_ses(history)

    forecast_dates = pd.date_range(
        start=test_start,
        end=test_end,
        freq="D"
    )

    forecast = pd.DataFrame({
        "Date": forecast_dates,
        "SES_Forecast": final_level
    })

    ses_forecasts[period] = forecast

    print(f"\n{period}")
    print("Forecast days:", len(forecast))
    print("Daily forecast:", round(final_level, 3))
    print("90-day forecast:", round(forecast["SES_Forecast"].sum(), 2))


In [ ]:

ses_evaluation_results = []

for period, dates in evaluation_periods.items():

    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])

    actual = stockout_sku_ts.loc[
        test_start:test_end,
        ["Units_Sold", "In_Stock_Flag"]
    ].copy()

    forecast = ses_forecasts[period].set_index("Date")

    evaluation = actual.join(forecast, how="inner")

    # Evaluate only days when the SKU was available
    evaluation = evaluation[evaluation["In_Stock_Flag"]].copy()

    evaluation["Error"] = (
        evaluation["SES_Forecast"] - evaluation["Units_Sold"]
    )

    actual_units = evaluation["Units_Sold"].sum()
    forecast_units = evaluation["SES_Forecast"].sum()

    mae = evaluation["Error"].abs().mean()
    wape = evaluation["Error"].abs().sum() / actual_units * 100
    bias = (forecast_units - actual_units) / actual_units * 100

    ses_evaluation_results.append({
        "Evaluation_Period": period,
        "Evaluated_Days": len(evaluation),
        "Actual_Units": actual_units,
        "Forecast_Units": round(forecast_units, 2),
        "MAE": round(mae, 2),
        "WAPE_Pct": round(wape, 2),
        "Bias_Pct": round(bias, 2)
    })

ses_evaluation_df = pd.DataFrame(ses_evaluation_results)

display(ses_evaluation_df)


In [ ]:

sku_daily_df = daily_sales_df[
    ["Date", "SKU_ID", "Units_Sold", "In_Stock_Flag"]
].copy()

sku_daily_df["Date"] = pd.to_datetime(sku_daily_df["Date"])

sku_daily_df["Sales_For_Model"] = (
    sku_daily_df["Units_Sold"]
    .where(sku_daily_df["In_Stock_Flag"], np.nan)
)

sku_daily_df = sku_daily_df.sort_values(
    ["SKU_ID", "Date"]
).reset_index(drop=True)

print("Total rows:", len(sku_daily_df))
print("Unique SKUs:", sku_daily_df["SKU_ID"].nunique())
print("Missing sales observations:", sku_daily_df["Sales_For_Model"].isna().sum())
print("Date range:", sku_daily_df["Date"].min(), "to", sku_daily_df["Date"].max())


In [ ]:

def generate_sku_forecasts(history, forecast_dates):

    # Moving Average
    moving_average = history.mean()

    # Weekly Seasonal Naive
    recent_28 = history.tail(28)
    weekday_forecasts = {}

    for weekday in range(7):
        weekday_values = recent_28[
            recent_28.index.dayofweek == weekday
        ].dropna()

        weekday_forecasts[weekday] = (
            weekday_values.iloc[-1]
            if not weekday_values.empty
            else moving_average
        )

    seasonal_naive = [
        weekday_forecasts[date.dayofweek]
        for date in forecast_dates
    ]

    # Missing-data-aware Exponential Smoothing
    observed = history.dropna()

    if len(observed) >= 2:
        alpha, final_level, _ = fit_missing_aware_ses(history)
    else:
        alpha = np.nan
        final_level = moving_average

    return pd.DataFrame({
        "Date": forecast_dates,
        "Moving_Average": moving_average,
        "Seasonal_Naive": seasonal_naive,
        "Exponential_Smoothing": final_level
    })


In [ ]:

test_forecasts = {}

for period, dates in evaluation_periods.items():

    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])

    history = stockout_sku_ts.loc[
        stockout_sku_ts.index < test_start,
        "Sales_For_Model"
    ]

    forecast_dates = pd.date_range(
        start=test_start,
        end=test_end,
        freq="D"
    )

    test_forecasts[period] = generate_sku_forecasts(
        history,
        forecast_dates
    )

    print(f"\n{period}")
    display(test_forecasts[period].head())
    print("Forecast days:", len(test_forecasts[period]))


In [ ]:

all_sku_forecasts = []

for period, dates in evaluation_periods.items():

    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])

    forecast_dates = pd.date_range(
        start=test_start,
        end=test_end,
        freq="D"
    )

    for sku_id, sku_data in sku_daily_df.groupby("SKU_ID"):

        history = (
            sku_data.loc[
                sku_data["Date"] < test_start
            ]
            .set_index("Date")["Sales_For_Model"]
            .asfreq("D")
        )

        forecasts = generate_sku_forecasts(
            history,
            forecast_dates
        )

        forecasts["SKU_ID"] = sku_id
        forecasts["Evaluation_Period"] = period

        all_sku_forecasts.append(forecasts)

all_sku_forecasts_df = pd.concat(
    all_sku_forecasts,
    ignore_index=True
)

print("Total forecast rows:", len(all_sku_forecasts_df))
print("Unique SKUs:", all_sku_forecasts_df["SKU_ID"].nunique())
print("\nRows by evaluation period:")
display(all_sku_forecasts_df.groupby("Evaluation_Period").size())


In [ ]:

actual_sales_eval = sku_daily_df[
    sku_daily_df["Date"].between(
        pd.Timestamp("2024-10-01"),
        pd.Timestamp("2025-12-29")
    )
].copy()

actual_sales_eval["Evaluation_Period"] = np.select(
    [
        actual_sales_eval["Date"].between(
            pd.Timestamp("2024-10-01"),
            pd.Timestamp("2024-12-29")
        ),
        actual_sales_eval["Date"].between(
            pd.Timestamp("2025-10-01"),
            pd.Timestamp("2025-12-29")
        )
    ],
    ["Peak_2024", "Peak_2025"],
    default=None
)

actual_sales_eval = actual_sales_eval[
    actual_sales_eval["Evaluation_Period"].notna()
    & actual_sales_eval["In_Stock_Flag"]
].copy()

print("Actual in-stock evaluation rows:", len(actual_sales_eval))
print("\nRows by evaluation period:")
display(actual_sales_eval.groupby("Evaluation_Period").size())


In [ ]:

forecast_evaluation_df = all_sku_forecasts_df.merge(
    actual_sales_eval[
        ["SKU_ID", "Date", "Evaluation_Period", "Units_Sold"]
    ],
    on=["SKU_ID", "Date", "Evaluation_Period"],
    how="inner"
)

print("Joined evaluation rows:", len(forecast_evaluation_df))
print("\nRows by evaluation period:")
display(
    forecast_evaluation_df.groupby("Evaluation_Period").size()
)

print("\nSample records:")
display(forecast_evaluation_df.head())


In [ ]:

def calculate_forecast_metrics(group, forecast_column):

    actual = group["Units_Sold"]
    forecast = group[forecast_column]

    actual_total = actual.sum()

    return pd.Series({
        "Evaluated_Days": len(group),
        "Actual_Units": actual_total,
        "Forecast_Units": forecast.sum(),
        "MAE": (forecast - actual).abs().mean(),
        "WAPE_Pct": (
            (forecast - actual).abs().sum() / actual_total * 100
            if actual_total > 0 else np.nan
        ),
        "Bias_Pct": (
            (forecast.sum() - actual_total) / actual_total * 100
            if actual_total > 0 else np.nan
        )
    })


metric_results = []

forecast_methods = {
    "Moving_Average": "Moving_Average",
    "Seasonal_Naive": "Seasonal_Naive",
    "Exponential_Smoothing": "Exponential_Smoothing"
}

for method, column in forecast_methods.items():

    method_metrics = (
        forecast_evaluation_df
        .groupby(["Evaluation_Period", "SKU_ID"])
        .apply(
            lambda group: calculate_forecast_metrics(group, column),
            include_groups=False
        )
        .reset_index()
    )

    method_metrics["Forecast_Method"] = method
    metric_results.append(method_metrics)

sku_forecast_metrics_df = pd.concat(
    metric_results,
    ignore_index=True
)

print("Total SKU-method evaluation records:", len(sku_forecast_metrics_df))
print("Unique SKUs:", sku_forecast_metrics_df["SKU_ID"].nunique())
print("\nRecords by method and evaluation period:")
display(
    sku_forecast_metrics_df.groupby(
        ["Evaluation_Period", "Forecast_Method"]
    ).size()
)


In [ ]:

overall_forecast_comparison = (
    forecast_evaluation_df
    .groupby("Evaluation_Period")
    .apply(
        lambda group: pd.Series({
            "Evaluated_SKU_Days": len(group),
            "Actual_Units": group["Units_Sold"].sum(),

            "Moving_Average_WAPE": (
                (group["Moving_Average"] - group["Units_Sold"]).abs().sum()
                / group["Units_Sold"].sum() * 100
            ),

            "Seasonal_Naive_WAPE": (
                (group["Seasonal_Naive"] - group["Units_Sold"]).abs().sum()
                / group["Units_Sold"].sum() * 100
            ),

            "Exponential_Smoothing_WAPE": (
                (group["Exponential_Smoothing"] - group["Units_Sold"]).abs().sum()
                / group["Units_Sold"].sum() * 100
            ),

            "Moving_Average_Bias": (
                (group["Moving_Average"].sum() - group["Units_Sold"].sum())
                / group["Units_Sold"].sum() * 100
            ),

            "Seasonal_Naive_Bias": (
                (group["Seasonal_Naive"].sum() - group["Units_Sold"].sum())
                / group["Units_Sold"].sum() * 100
            ),

            "Exponential_Smoothing_Bias": (
                (group["Exponential_Smoothing"].sum() - group["Units_Sold"].sum())
                / group["Units_Sold"].sum() * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(overall_forecast_comparison.round(2))


In [ ]:

def generate_sku_forecasts(history, forecast_dates):

    # Use the previous 28 calendar days
    recent_28 = history.tail(28)

    moving_average = recent_28.mean()

    # Fallback if there are no available observations
    if pd.isna(moving_average):
        moving_average = history.mean()

    # Weekly Seasonal Naive
    weekday_forecasts = {}

    for weekday in range(7):
        weekday_values = recent_28[
            recent_28.index.dayofweek == weekday
        ].dropna()

        weekday_forecasts[weekday] = (
            weekday_values.iloc[-1]
            if not weekday_values.empty
            else moving_average
        )

    seasonal_naive = [
        weekday_forecasts[date.dayofweek]
        for date in forecast_dates
    ]

    # Missing-data-aware Exponential Smoothing
    observed = history.dropna()

    if len(observed) >= 2:
        alpha, final_level, _ = fit_missing_aware_ses(history)
    else:
        final_level = moving_average

    return pd.DataFrame({
        "Date": forecast_dates,
        "Moving_Average": moving_average,
        "Seasonal_Naive": seasonal_naive,
        "Exponential_Smoothing": final_level
    })


In [ ]:

all_sku_forecasts = []

for period, dates in evaluation_periods.items():

    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])

    forecast_dates = pd.date_range(
        start=test_start,
        end=test_end,
        freq="D"
    )

    for sku_id, sku_data in sku_daily_df.groupby("SKU_ID"):

        history = (
            sku_data.loc[
                sku_data["Date"] < test_start
            ]
            .set_index("Date")["Sales_For_Model"]
            .asfreq("D")
        )

        forecasts = generate_sku_forecasts(
            history,
            forecast_dates
        )

        forecasts["SKU_ID"] = sku_id
        forecasts["Evaluation_Period"] = period

        all_sku_forecasts.append(forecasts)

all_sku_forecasts_df = pd.concat(
    all_sku_forecasts,
    ignore_index=True
)

print("Total forecast rows:", len(all_sku_forecasts_df))
print("Unique SKUs:", all_sku_forecasts_df["SKU_ID"].nunique())


In [ ]:

forecast_evaluation_df = all_sku_forecasts_df.merge(
    actual_sales_eval[
        ["SKU_ID", "Date", "Evaluation_Period", "Units_Sold"]
    ],
    on=["SKU_ID", "Date", "Evaluation_Period"],
    how="inner"
)

print("Joined evaluation rows:", len(forecast_evaluation_df))

print("\nRows by evaluation period:")
display(
    forecast_evaluation_df.groupby("Evaluation_Period").size()
)


In [ ]:

overall_forecast_comparison = (
    forecast_evaluation_df
    .groupby("Evaluation_Period")
    .apply(
        lambda group: pd.Series({
            "Evaluated_SKU_Days": len(group),
            "Actual_Units": group["Units_Sold"].sum(),

            "Moving_Average_WAPE": (
                (group["Moving_Average"] - group["Units_Sold"]).abs().sum()
                / group["Units_Sold"].sum() * 100
            ),

            "Seasonal_Naive_WAPE": (
                (group["Seasonal_Naive"] - group["Units_Sold"]).abs().sum()
                / group["Units_Sold"].sum() * 100
            ),

            "Exponential_Smoothing_WAPE": (
                (group["Exponential_Smoothing"] - group["Units_Sold"]).abs().sum()
                / group["Units_Sold"].sum() * 100
            ),

            "Moving_Average_Bias": (
                (group["Moving_Average"].sum() - group["Units_Sold"].sum())
                / group["Units_Sold"].sum() * 100
            ),

            "Seasonal_Naive_Bias": (
                (group["Seasonal_Naive"].sum() - group["Units_Sold"].sum())
                / group["Units_Sold"].sum() * 100
            ),

            "Exponential_Smoothing_Bias": (
                (group["Exponential_Smoothing"].sum() - group["Units_Sold"].sum())
                / group["Units_Sold"].sum() * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(overall_forecast_comparison.round(2))


In [ ]:

metric_results = []

forecast_methods = {
    "Moving_Average": "Moving_Average",
    "Seasonal_Naive": "Seasonal_Naive",
    "Exponential_Smoothing": "Exponential_Smoothing"
}

for method, column in forecast_methods.items():

    method_metrics = (
        forecast_evaluation_df
        .groupby(["Evaluation_Period", "SKU_ID"])
        .apply(
            lambda group: calculate_forecast_metrics(group, column),
            include_groups=False
        )
        .reset_index()
    )

    method_metrics["Forecast_Method"] = method
    metric_results.append(method_metrics)

sku_forecast_metrics_df = pd.concat(
    metric_results,
    ignore_index=True
)

print("Total records:", len(sku_forecast_metrics_df))

display(
    sku_forecast_metrics_df.head()
)


In [ ]:

# Compare the number of SKUs where each method performs best

sku_method_wins = (
    sku_forecast_metrics_df
    .groupby(["Evaluation_Period", "SKU_ID"])
    .apply(
        lambda group: pd.Series({
            "Best_MAE_Method": group.loc[
                group["MAE"].idxmin(), "Forecast_Method"
            ],
            "Best_WAPE_Method": (
                group.loc[group["WAPE_Pct"].idxmin(), "Forecast_Method"]
                if group["WAPE_Pct"].notna().any()
                else "Not Available"
            )
        }),
        include_groups=False
    )
    .reset_index()
)

print("Best method by MAE:")
display(
    sku_method_wins
    .groupby(["Evaluation_Period", "Best_MAE_Method"])
    .size()
    .unstack(fill_value=0)
)

print("Best method by WAPE:")
display(
    sku_method_wins
    .groupby(["Evaluation_Period", "Best_WAPE_Method"])
    .size()
    .unstack(fill_value=0)
)


In [ ]:

sku_profile_metrics_df = sku_forecast_metrics_df.merge(
    sku_demand_profile[
        [
            "SKU_ID",
            "Demand_Variability_Type",
            "Trend_Classification",
            "Seasonality_Classification"
        ]
    ],
    on="SKU_ID",
    how="left"
)

profile_method_comparison = (
    sku_profile_metrics_df
    .groupby(
        [
            "Evaluation_Period",
            "Demand_Variability_Type",
            "Forecast_Method"
        ]
    )
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Average_WAPE=("WAPE_Pct", "mean"),
        Median_WAPE=("WAPE_Pct", "median"),
        Average_Bias=("Bias_Pct", "mean")
    )
    .reset_index()
)

display(
    profile_method_comparison.round(2)
)


In [ ]:

trend_method_comparison = (
    sku_profile_metrics_df
    .groupby(
        [
            "Evaluation_Period",
            "Trend_Classification",
            "Forecast_Method"
        ]
    )
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Average_WAPE=("WAPE_Pct", "mean"),
        Median_WAPE=("WAPE_Pct", "median"),
        Average_Bias=("Bias_Pct", "mean")
    )
    .reset_index()
)

display(
    trend_method_comparison.round(2)
)


In [ ]:

seasonality_method_comparison = (
    sku_profile_metrics_df
    .groupby(
        [
            "Evaluation_Period",
            "Seasonality_Classification",
            "Forecast_Method"
        ]
    )
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Average_WAPE=("WAPE_Pct", "mean"),
        Median_WAPE=("WAPE_Pct", "median"),
        Average_Bias=("Bias_Pct", "mean")
    )
    .reset_index()
)

display(
    seasonality_method_comparison.round(2)
)


In [ ]:

def fit_calendar_aware_ses(series):
    values = series.to_numpy(dtype=float)
    valid_positions = np.flatnonzero(~np.isnan(values))

    if len(valid_positions) < 2:
        raise ValueError("Insufficient observations.")

    def calculate_sse(alpha):
        level = values[valid_positions[0]]
        previous_position = valid_positions[0]
        errors = []

        for position in valid_positions[1:]:
            gap = position - previous_position

            # Carry the previous level forward across missing days
            adjusted_level = level

            errors.append(values[position] - adjusted_level)

            # Update using the elapsed calendar time
            effective_alpha = 1 - (1 - alpha) ** gap
            level = (
                effective_alpha * values[position]
                + (1 - effective_alpha) * level
            )

            previous_position = position

        return np.sum(np.square(errors))

    result = minimize_scalar(
        calculate_sse,
        bounds=(0.0001, 0.9999),
        method="bounded"
    )

    alpha = result.x
    level = values[valid_positions[0]]
    previous_position = valid_positions[0]

    for position in valid_positions[1:]:
        gap = position - previous_position
        effective_alpha = 1 - (1 - alpha) ** gap

        level = (
            effective_alpha * values[position]
            + (1 - effective_alpha) * level
        )

        previous_position = position

    return alpha, level, result.fun


In [ ]:

ses_comparison_results = []

for period, dates in evaluation_periods.items():

    train_end = pd.Timestamp(dates["train_end"])

    history = (
        stockout_sku_ts.loc[
            stockout_sku_ts.index <= train_end,
            "Sales_For_Model"
        ]
        .asfreq("D")
    )

    # Original missing-data-aware SES
    original_alpha, original_level, _ = fit_missing_aware_ses(history)

    # Calendar-aware SES
    calendar_alpha, calendar_level, _ = fit_calendar_aware_ses(history)

    ses_comparison_results.append({
        "Evaluation_Period": period,
        "Original_Alpha": original_alpha,
        "Original_Forecast_Daily": original_level,
        "Calendar_Aware_Alpha": calendar_alpha,
        "Calendar_Aware_Forecast_Daily": calendar_level,
        "Original_90D_Forecast": original_level * 90,
        "Calendar_Aware_90D_Forecast": calendar_level * 90
    })

ses_comparison_df = pd.DataFrame(ses_comparison_results)

display(ses_comparison_df.round(3))


In [ ]:

calendar_ses_evaluation = []

for period, dates in evaluation_periods.items():

    train_end = pd.Timestamp(dates["train_end"])
    test_start = pd.Timestamp(dates["test_start"])
    test_end = pd.Timestamp(dates["test_end"])

    history = (
        stockout_sku_ts.loc[
            stockout_sku_ts.index <= train_end,
            "Sales_For_Model"
        ].asfreq("D")
    )

    test_data = stockout_sku_ts.loc[
        (stockout_sku_ts.index >= test_start) &
        (stockout_sku_ts.index <= test_end)
    ].copy()

    original_alpha, original_level, _ = fit_missing_aware_ses(history)
    calendar_alpha, calendar_level, _ = fit_calendar_aware_ses(history)

    actual = test_data.loc[
        test_data["In_Stock_Flag"], "Units_Sold"
    ]

    for method, forecast_level in [
        ("Original_SES", original_level),
        ("Calendar_Aware_SES", calendar_level)
    ]:

        forecast = np.full(len(actual), forecast_level)
        actual_values = actual.to_numpy()

        actual_total = actual_values.sum()
        forecast_total = forecast.sum()

        calendar_ses_evaluation.append({
            "Evaluation_Period": period,
            "Forecast_Method": method,
            "Evaluated_Days": len(actual_values),
            "Actual_Units": actual_total,
            "Forecast_Units": forecast_total,
            "MAE": np.mean(np.abs(forecast - actual_values)),
            "WAPE_Pct": (
                np.sum(np.abs(forecast - actual_values))
                / actual_total * 100
            ),
            "Bias_Pct": (
                (forecast_total - actual_total)
                / actual_total * 100
            )
        })

calendar_ses_evaluation_df = pd.DataFrame(
    calendar_ses_evaluation
)

display(calendar_ses_evaluation_df.round(2))


In [ ]:

# Set the final forecast horizon
forecast_start = pd.Timestamp("2026-01-01")
forecast_end = pd.Timestamp("2026-03-31")

final_forecast_dates = pd.date_range(
    start=forecast_start,
    end=forecast_end,
    freq="D"
)

# Prepare complete historical sales series for each SKU
final_sku_history = {}

for sku, group in sku_daily_df.groupby("SKU_ID"):

    history = (
        group
        .set_index("Date")["Sales_For_Model"]
        .sort_index()
        .asfreq("D")
    )

    final_sku_history[sku] = history

print("Number of SKUs:", len(final_sku_history))
print("Historical start:", sku_daily_df["Date"].min())
print("Historical end:", sku_daily_df["Date"].max())
print("Forecast start:", final_forecast_dates.min())
print("Forecast end:", final_forecast_dates.max())
print("Forecast days:", len(final_forecast_dates))


In [ ]:

final_forecast_results = []

for sku, history in final_sku_history.items():

    sku_forecast = generate_sku_forecasts(
        history,
        final_forecast_dates
    )

    sku_forecast["SKU_ID"] = sku

    final_forecast_results.append(sku_forecast)

final_sku_forecasts_df = pd.concat(
    final_forecast_results,
    ignore_index=True
)

# Arrange columns
final_sku_forecasts_df = final_sku_forecasts_df[
    [
        "Date",
        "SKU_ID",
        "Moving_Average",
        "Seasonal_Naive",
        "Exponential_Smoothing"
    ]
]

print("Total forecast records:", len(final_sku_forecasts_df))
print("Unique SKUs:", final_sku_forecasts_df["SKU_ID"].nunique())
print("Forecast dates:", final_sku_forecasts_df["Date"].nunique())

display(final_sku_forecasts_df.head(10))


In [ ]:

# Add product category information
final_sku_forecasts_df = final_sku_forecasts_df.merge(
    products_df[["SKU_ID", "Category"]],
    on="SKU_ID",
    how="left"
)

# Aggregate forecasts by category and method
category_forecast_comparison = (
    final_sku_forecasts_df
    .groupby("Category")
    .agg(
        Moving_Average_Units=("Moving_Average", "sum"),
        Seasonal_Naive_Units=("Seasonal_Naive", "sum"),
        Exponential_Smoothing_Units=("Exponential_Smoothing", "sum")
    )
    .reset_index()
)

# Add overall total
overall_totals = pd.DataFrame([{
    "Category": "TOTAL",
    "Moving_Average_Units": category_forecast_comparison["Moving_Average_Units"].sum(),
    "Seasonal_Naive_Units": category_forecast_comparison["Seasonal_Naive_Units"].sum(),
    "Exponential_Smoothing_Units": category_forecast_comparison["Exponential_Smoothing_Units"].sum()
}])

category_forecast_comparison = pd.concat(
    [category_forecast_comparison, overall_totals],
    ignore_index=True
)

display(category_forecast_comparison.round(0))


In [ ]:

category_forecast_summary = (
    final_sku_forecasts_df
    .groupby("Category")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Forecast_Units=("Exponential_Smoothing", "sum"),
        Average_SKU_Forecast=("Exponential_Smoothing", "mean"),
        Minimum_SKU_Forecast=("Exponential_Smoothing", "min"),
        Maximum_SKU_Forecast=("Exponential_Smoothing", "max")
    )
    .reset_index()
)

category_forecast_summary["Average_Daily_Category_Demand"] = (
    category_forecast_summary["Forecast_Units"] / 90
)

display(category_forecast_summary.round(2))


In [ ]:
print("PURCHASE ORDERS")
print(purchase_orders_df.columns.tolist())
display(purchase_orders_df.head(5))

print("\nPURCHASE ORDER RECEIPTS")
print(purchase_order_receipts_df.columns.tolist())
display(purchase_order_receipts_df.head(5))

print("\nDAILY SALES – FINAL DATE")
display(
    daily_sales_df[
        daily_sales_df["Date"] == daily_sales_df["Date"].max()
    ].head(5)
)

In [ ]:

# Set the inventory planning cut-off date
planning_date = pd.Timestamp("2025-12-31")

# Check source data date ranges
print("Purchase Order Date Range:")
print(purchase_orders_df["PO_Date"].min(),
      "to",
      purchase_orders_df["PO_Date"].max())

print("\nReceipt Date Range:")
print(purchase_order_receipts_df["Receipt_Date"].min(),
      "to",
      purchase_order_receipts_df["Receipt_Date"].max())

# Identify purchase orders placed by the planning date
po_as_of_date = purchase_orders_df[
    pd.to_datetime(purchase_orders_df["PO_Date"]) <= planning_date
].copy()

# Identify receipts received by the planning date
receipts_as_of_date = purchase_order_receipts_df[
    pd.to_datetime(purchase_order_receipts_df["Receipt_Date"]) <= planning_date
].copy()

# Calculate outstanding quantities as of the planning date
received_by_po = (
    receipts_as_of_date
    .groupby("PO_ID")["Received_Quantity"]
    .sum()
)

po_as_of_date["Received_As_Of_Date"] = (
    po_as_of_date["PO_ID"].map(received_by_po).fillna(0)
)

po_as_of_date["Outstanding_As_Of_Date"] = (
    po_as_of_date["Order_Quantity"]
    - po_as_of_date["Received_As_Of_Date"]
).clip(lower=0)

print("\nPurchase Order Position as of", planning_date.date())
print("Purchase orders:", len(po_as_of_date))
print("Outstanding PO lines:",
      (po_as_of_date["Outstanding_As_Of_Date"] > 0).sum())
print("Outstanding units:",
      po_as_of_date["Outstanding_As_Of_Date"].sum())


In [ ]:

# Inventory position on the final day of historical data
closing_inventory_df = (
    daily_sales_df[
        pd.to_datetime(daily_sales_df["Date"]) == planning_date
    ][["SKU_ID", "Closing_Inventory"]]
    .copy()
)

# Aggregate outstanding purchase orders by SKU
outstanding_po_by_sku = (
    po_as_of_date[
        po_as_of_date["Outstanding_As_Of_Date"] > 0
    ]
    .groupby("SKU_ID")
    .agg(
        Outstanding_PO_Units=("Outstanding_As_Of_Date", "sum"),
        Outstanding_PO_Count=("PO_ID", "nunique"),
        Earliest_Expected_Delivery=(
            "Expected_Delivery_Date", "min"
        ),
        Latest_Expected_Delivery=(
            "Expected_Delivery_Date", "max"
        )
    )
    .reset_index()
)

# Combine inventory and outstanding orders
sku_stock_position_df = (
    products_df[["SKU_ID", "Category"]]
    .merge(closing_inventory_df, on="SKU_ID", how="left")
    .merge(outstanding_po_by_sku, on="SKU_ID", how="left")
)

# Replace missing outstanding quantities with zero
sku_stock_position_df["Outstanding_PO_Units"] = (
    sku_stock_position_df["Outstanding_PO_Units"].fillna(0)
)

sku_stock_position_df["Outstanding_PO_Count"] = (
    sku_stock_position_df["Outstanding_PO_Count"].fillna(0)
)

sku_stock_position_df["Closing_Inventory"] = (
    sku_stock_position_df["Closing_Inventory"].fillna(0)
)

# Current stock plus stock already on order
sku_stock_position_df["Inventory_And_Inbound_Units"] = (
    sku_stock_position_df["Closing_Inventory"]
    + sku_stock_position_df["Outstanding_PO_Units"]
)

print("SKU records:", len(sku_stock_position_df))
print("\nTotal closing inventory:",
      sku_stock_position_df["Closing_Inventory"].sum())

print("Total outstanding PO units:",
      sku_stock_position_df["Outstanding_PO_Units"].sum())

print("Total inventory and inbound units:",
      sku_stock_position_df["Inventory_And_Inbound_Units"].sum())

display(
    sku_stock_position_df
    .sort_values("Outstanding_PO_Units", ascending=False)
    .head(10)
)


In [ ]:

# Classify outstanding purchase orders by expected delivery date
outstanding_po_detail = po_as_of_date[
    po_as_of_date["Outstanding_As_Of_Date"] > 0
].copy()

outstanding_po_detail["Expected_Delivery_Date"] = pd.to_datetime(
    outstanding_po_detail["Expected_Delivery_Date"]
)

outstanding_po_detail["Delivery_Status_As_Of_Date"] = np.select(
    [
        outstanding_po_detail["Expected_Delivery_Date"] < planning_date,
        outstanding_po_detail["Expected_Delivery_Date"] == planning_date,
        outstanding_po_detail["Expected_Delivery_Date"] > planning_date
    ],
    [
        "Overdue",
        "Due on Planning Date",
        "Future Expected Delivery"
    ],
    default="Unknown"
)

po_delivery_summary = (
    outstanding_po_detail
    .groupby("Delivery_Status_As_Of_Date")
    .agg(
        PO_Lines=("PO_ID", "nunique"),
        SKUs=("SKU_ID", "nunique"),
        Outstanding_Units=("Outstanding_As_Of_Date", "sum")
    )
    .reset_index()
)

display(po_delivery_summary)

print("\nSample overdue outstanding orders:")
display(
    outstanding_po_detail[
        outstanding_po_detail["Delivery_Status_As_Of_Date"] == "Overdue"
    ][
        ["PO_ID", "SKU_ID", "Outstanding_As_Of_Date",
         "Expected_Delivery_Date", "PO_Status"]
    ]
    .head(10)
)


In [ ]:

# Separate inbound quantities by delivery status at SKU level
inbound_by_sku = (
    outstanding_po_detail
    .pivot_table(
        index="SKU_ID",
        columns="Delivery_Status_As_Of_Date",
        values="Outstanding_As_Of_Date",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Ensure all expected columns exist
expected_columns = [
    "Overdue",
    "Due on Planning Date",
    "Future Expected Delivery"
]

for col in expected_columns:
    if col not in inbound_by_sku.columns:
        inbound_by_sku[col] = 0

inbound_by_sku = inbound_by_sku.rename(columns={
    "Overdue": "Overdue_PO_Units",
    "Due on Planning Date": "Due_On_Cutoff_Units",
    "Future Expected Delivery": "Future_Inbound_Units"
})

# Merge the inbound breakdown with the stock position
sku_stock_position_df = (
    sku_stock_position_df
    .drop(columns=[
        "Outstanding_PO_Units",
        "Outstanding_PO_Count",
        "Earliest_Expected_Delivery",
        "Latest_Expected_Delivery",
        "Inventory_And_Inbound_Units"
    ])
    .merge(inbound_by_sku, on="SKU_ID", how="left")
)

inbound_columns = [
    "Overdue_PO_Units",
    "Due_On_Cutoff_Units",
    "Future_Inbound_Units"
]

sku_stock_position_df[inbound_columns] = (
    sku_stock_position_df[inbound_columns].fillna(0)
)

# Conservative stock position: existing stock + future-dated inbound
sku_stock_position_df["Planning_Stock_Units"] = (
    sku_stock_position_df["Closing_Inventory"]
    + sku_stock_position_df["Future_Inbound_Units"]
)

print("SKU records:", len(sku_stock_position_df))
print("\nInbound breakdown:")
display(
    sku_stock_position_df[
        ["Closing_Inventory"] + inbound_columns + ["Planning_Stock_Units"]
    ].sum().to_frame("Total_Units")
)

display(
    sku_stock_position_df
    .sort_values("Overdue_PO_Units", ascending=False)
    .head(10)
)


In [ ]:

print("SUPPLIERS")
print("Columns:", suppliers_df.columns.tolist())
display(suppliers_df.head())

print("\nPRODUCT-SUPPLIER MAPPING")
print("Columns:", product_supplier_mapping_df.columns.tolist())
display(product_supplier_mapping_df.head())


In [ ]:

# List DataFrame variables currently available in the notebook
dataframe_variables = {
    name: value.shape
    for name, value in globals().items()
    if isinstance(value, pd.DataFrame)
}

display(dataframe_variables)



In [ ]:

print("PRODUCT-SUPPLIER MAPPING")
print("Columns:", product_supplier_df.columns.tolist())
display(product_supplier_df.head(10))

print("\nSupplier mapping counts:")
display(
    product_supplier_df
    .groupby("Supplier_ID")
    .size()
    .reset_index(name="Mapped_SKUs")
    .sort_values("Mapped_SKUs", ascending=False)
)


In [ ]:

# Select preferred supplier for each SKU
preferred_supplier_df = (
    product_supplier_df[
        product_supplier_df["Supplier_Role"] == "Preferred"
    ]
    [["SKU_ID", "Supplier_ID"]]
    .copy()
)

# Attach supplier lead time and reliability
preferred_supplier_df = preferred_supplier_df.merge(
    suppliers_df[
        ["Supplier_ID", "Supplier_Name",
         "Lead_Time_Days", "Supplier_Reliability"]
    ],
    on="Supplier_ID",
    how="left"
)

# Add preferred supplier details to SKU stock position
sku_stock_position_df = sku_stock_position_df.merge(
    preferred_supplier_df,
    on="SKU_ID",
    how="left"
)

print("SKU records:", len(sku_stock_position_df))
print("SKUs with preferred supplier:",
      sku_stock_position_df["Supplier_ID"].nunique())

print("\nSupplier lead-time summary:")
display(
    sku_stock_position_df["Lead_Time_Days"]
    .describe()
    .round(1)
)

print("\nSupplier reliability distribution:")
display(
    sku_stock_position_df["Supplier_Reliability"]
    .value_counts()
)

display(
    sku_stock_position_df[
        ["SKU_ID", "Category", "Supplier_ID",
         "Lead_Time_Days", "Supplier_Reliability",
         "Closing_Inventory", "Future_Inbound_Units"]
    ].head(10)
)


In [ ]:

# Calculate expected daily demand for each SKU
sku_forecast_rate = (
    final_sku_forecasts_df
    .groupby("SKU_ID")
    .agg(
        Forecast_Daily_Demand=(
            "Exponential_Smoothing", "mean"
        )
    )
    .reset_index()
)

# Add forecast rate to the stock position
sku_stock_position_df = sku_stock_position_df.merge(
    sku_forecast_rate,
    on="SKU_ID",
    how="left"
)

# Calculate demand during supplier lead time
sku_stock_position_df["Lead_Time_Demand"] = (
    sku_stock_position_df["Forecast_Daily_Demand"]
    * sku_stock_position_df["Lead_Time_Days"]
)

# Safety stock assumption
sku_stock_position_df["Safety_Stock_Units"] = 2

# Initial lead-time stock requirement
sku_stock_position_df["Lead_Time_Stock_Requirement"] = (
    sku_stock_position_df["Lead_Time_Demand"]
    + sku_stock_position_df["Safety_Stock_Units"]
)

print("SKU records:", len(sku_stock_position_df))

display(
    sku_stock_position_df[
        [
            "SKU_ID",
            "Forecast_Daily_Demand",
            "Lead_Time_Days",
            "Lead_Time_Demand",
            "Safety_Stock_Units",
            "Lead_Time_Stock_Requirement"
        ]
    ]
    .head(10)
    .round(2)
)


In [ ]:

# Calculate the expected arrival date of a new order for each SKU
sku_stock_position_df["New_Order_Arrival_Date"] = (
    planning_date
    + pd.to_timedelta(
        sku_stock_position_df["Lead_Time_Days"],
        unit="D"
    )
)

# Keep only future-dated outstanding orders
future_po_detail = outstanding_po_detail[
    outstanding_po_detail["Delivery_Status_As_Of_Date"]
    == "Future Expected Delivery"
].copy()

# Attach each SKU's expected new-order arrival date
future_po_detail = future_po_detail.merge(
    sku_stock_position_df[
        ["SKU_ID", "New_Order_Arrival_Date"]
    ],
    on="SKU_ID",
    how="left"
)

# Identify existing orders expected to arrive within supplier lead time
inbound_within_lead_time = (
    future_po_detail[
        future_po_detail["Expected_Delivery_Date"]
        <= future_po_detail["New_Order_Arrival_Date"]
    ]
    .groupby("SKU_ID")["Outstanding_As_Of_Date"]
    .sum()
    .rename("Inbound_Within_Lead_Time")
    .reset_index()
)

# Add the inbound quantities to the SKU planning table
sku_stock_position_df = sku_stock_position_df.merge(
    inbound_within_lead_time,
    on="SKU_ID",
    how="left"
)

sku_stock_position_df["Inbound_Within_Lead_Time"] = (
    sku_stock_position_df["Inbound_Within_Lead_Time"].fillna(0)
)

# Calculate projected stock when a new order would arrive
sku_stock_position_df["Projected_Stock_At_New_Order_Arrival"] = (
    sku_stock_position_df["Closing_Inventory"]
    + sku_stock_position_df["Inbound_Within_Lead_Time"]
    - sku_stock_position_df["Lead_Time_Demand"]
)

# Compare projected stock with safety stock
sku_stock_position_df["Lead_Time_Stock_Gap"] = (
    sku_stock_position_df["Projected_Stock_At_New_Order_Arrival"]
    - sku_stock_position_df["Safety_Stock_Units"]
)

print("SKUs:", len(sku_stock_position_df))
print(
    "Inbound units expected within lead time:",
    sku_stock_position_df["Inbound_Within_Lead_Time"].sum()
)

display(
    sku_stock_position_df[
        [
            "SKU_ID",
            "Closing_Inventory",
            "Lead_Time_Demand",
            "Inbound_Within_Lead_Time",
            "Projected_Stock_At_New_Order_Arrival",
            "Safety_Stock_Units",
            "Lead_Time_Stock_Gap"
        ]
    ]
    .sort_values("Lead_Time_Stock_Gap")
    .head(15)
    .round(2)
)


In [ ]:

# Classify SKUs based on projected lead-time stock gap
sku_stock_position_df["Lead_Time_Risk"] = np.where(
    sku_stock_position_df["Lead_Time_Stock_Gap"] < 0,
    "Below Safety Stock",
    "Adequate"
)

# Summarise SKU counts and quantities by risk category
lead_time_risk_summary = (
    sku_stock_position_df
    .groupby("Lead_Time_Risk")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Closing_Inventory=("Closing_Inventory", "sum"),
        Future_Inbound=("Inbound_Within_Lead_Time", "sum"),
        Overdue_PO_Units=("Overdue_PO_Units", "sum"),
        Lead_Time_Demand=("Lead_Time_Demand", "sum")
    )
    .reset_index()
)

display(lead_time_risk_summary.round(2))

# Identify the SKUs with the largest lead-time gaps
print("\nTop 15 SKUs with the largest lead-time stock gaps:")

display(
    sku_stock_position_df[
        [
            "SKU_ID",
            "Category",
            "Supplier_ID",
            "Supplier_Reliability",
            "Lead_Time_Days",
            "Closing_Inventory",
            "Future_Inbound_Units",
            "Overdue_PO_Units",
            "Lead_Time_Demand",
            "Lead_Time_Stock_Gap"
        ]
    ]
    .sort_values("Lead_Time_Stock_Gap")
    .head(15)
    .round(2)
)


In [ ]:

# Classify whether each SKU has overdue purchase orders
sku_stock_position_df["Overdue_PO_Status"] = np.where(
    sku_stock_position_df["Overdue_PO_Units"] > 0,
    "Has Overdue PO",
    "No Overdue PO"
)

# Summarise lead-time risk and overdue PO position
replenishment_risk_summary = (
    sku_stock_position_df
    .groupby(["Lead_Time_Risk", "Overdue_PO_Status"])
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Closing_Inventory=("Closing_Inventory", "sum"),
        Lead_Time_Demand=("Lead_Time_Demand", "sum"),
        Overdue_PO_Units=("Overdue_PO_Units", "sum"),
        Future_Inbound_Units=("Inbound_Within_Lead_Time", "sum")
    )
    .reset_index()
)

display(replenishment_risk_summary.round(2))


In [ ]:
priority_sku_review = (
    sku_stock_position_df[
        sku_stock_position_df["Lead_Time_Risk"] == "Below Safety Stock"
    ]
    .sort_values("Lead_Time_Stock_Gap")
    [[
        "SKU_ID",
        "Category",
        "Closing_Inventory",
        "Lead_Time_Demand",
        "Inbound_Within_Lead_Time",
        "Lead_Time_Stock_Gap",
        "Overdue_PO_Units",
        "Due_On_Cutoff_Units",
        "Future_Inbound_Units",
        "Supplier_Name",
        "Lead_Time_Days",
        "Supplier_Reliability"
    ]]
    .head(20)
)

display(priority_sku_review.round(2))

In [ ]:
sku_stock_position_df["PO_Exception_Status"] = np.select(
    [
        sku_stock_position_df["Overdue_PO_Units"] > 0,
        sku_stock_position_df["Due_On_Cutoff_Units"] > 0
    ],
    [
        "Has Overdue PO",
        "Due on Planning Date"
    ],
    default="No Overdue or Due PO"
)

po_exception_risk_summary = (
    sku_stock_position_df
    .groupby(["Lead_Time_Risk", "PO_Exception_Status"])
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Closing_Inventory=("Closing_Inventory", "sum"),
        Lead_Time_Demand=("Lead_Time_Demand", "sum"),
        Overdue_PO_Units=("Overdue_PO_Units", "sum"),
        Due_On_Cutoff_Units=("Due_On_Cutoff_Units", "sum"),
        Future_Inbound_Units=("Inbound_Within_Lead_Time", "sum")
    )
    .reset_index()
)

display(po_exception_risk_summary.round(2))


In [ ]:
new_order_candidates_df = (
    sku_stock_position_df[
        (sku_stock_position_df["Lead_Time_Risk"] == "Below Safety Stock") &
        (sku_stock_position_df["PO_Exception_Status"] == "No Overdue or Due PO")
    ]
    .copy()
    .sort_values("Lead_Time_Stock_Gap")
)

candidate_columns = [
    "SKU_ID",
    "Category",
    "Closing_Inventory",
    "Lead_Time_Demand",
    "Inbound_Within_Lead_Time",
    "Lead_Time_Stock_Gap",
    "Future_Inbound_Units",
    "Supplier_Name",
    "Lead_Time_Days",
    "Supplier_Reliability"
]

display(
    new_order_candidates_df[candidate_columns]
    .head(20)
    .round(2)
)

In [ ]:
# Aggregate the selected SES forecast by SKU for the 90-day horizon
forecast_90d_by_sku = (
    final_sku_forecasts_df
    .groupby("SKU_ID")
    .agg(
        Forecast_90D_Units=("Exponential_Smoothing", "sum")
    )
    .reset_index()
)

# Identify outstanding future POs expected during the 90-day horizon
future_po_90d = future_po_detail[
    future_po_detail["Expected_Delivery_Date"] <= pd.Timestamp("2026-03-31")
].copy()

future_inbound_90d_by_sku = (
    future_po_90d
    .groupby("SKU_ID")
    .agg(
        Future_Inbound_90D=("Outstanding_As_Of_Date", "sum")
    )
    .reset_index()
)

# Combine stock, inbound and forecast demand
new_order_candidates_df = (
    new_order_candidates_df
    .drop(columns=["Forecast_90D_Units", "Future_Inbound_90D"], errors="ignore")
    .merge(forecast_90d_by_sku, on="SKU_ID", how="left")
    .merge(future_inbound_90d_by_sku, on="SKU_ID", how="left")
)

new_order_candidates_df["Future_Inbound_90D"] = (
    new_order_candidates_df["Future_Inbound_90D"].fillna(0)
)

new_order_candidates_df["Projected_90D_Stock_Balance"] = (
    new_order_candidates_df["Closing_Inventory"]
    + new_order_candidates_df["Future_Inbound_90D"]
    - new_order_candidates_df["Forecast_90D_Units"]
)

display(
    new_order_candidates_df[
        [
            "SKU_ID",
            "Category",
            "Closing_Inventory",
            "Future_Inbound_90D",
            "Forecast_90D_Units",
            "Projected_90D_Stock_Balance",
            "Supplier_Name",
            "Lead_Time_Days",
            "Supplier_Reliability"
        ]
    ]
    .sort_values("Projected_90D_Stock_Balance")
    .head(20)
    .round(2)
)

In [ ]:
# Prepare daily forecast by SKU
daily_forecast_lookup = {
    sku: group.set_index("Date")["Exponential_Smoothing"].sort_index()
    for sku, group in final_sku_forecasts_df.groupby("SKU_ID")
}

# Prepare future inbound quantities by SKU and expected delivery date
future_receipts = future_po_detail.copy()
future_receipts["Expected_Delivery_Date"] = pd.to_datetime(
    future_receipts["Expected_Delivery_Date"]
).dt.normalize()

future_receipts = future_receipts[
    future_receipts["Expected_Delivery_Date"].between(
        forecast_start, forecast_end
    )
]

receipt_lookup = {
    sku: group.groupby("Expected_Delivery_Date")[
        "Outstanding_As_Of_Date"
    ].sum()
    for sku, group in future_receipts.groupby("SKU_ID")
}

# Simulate daily stock position for the 134 candidates
projection_results = []

for row in new_order_candidates_df.itertuples(index=False):
    sku = row.SKU_ID
    stock = float(row.Closing_Inventory)

    daily_demand = daily_forecast_lookup[sku]
    daily_receipts = receipt_lookup.get(sku, pd.Series(dtype=float))

    first_shortage_date = pd.NaT
    minimum_stock = stock

    for date in final_forecast_dates:
        # Add expected receipts at the start of the delivery date
        stock += daily_receipts.get(date, 0)

        # Subtract forecast demand
        stock -= daily_demand.loc[date]

        minimum_stock = min(minimum_stock, stock)

        if stock < 0 and pd.isna(first_shortage_date):
            first_shortage_date = date

    projection_results.append({
        "SKU_ID": sku,
        "First_Projected_Shortage_Date": first_shortage_date,
        "Projected_Stock_31_Mar": stock,
        "Minimum_Projected_Stock": minimum_stock
    })

time_phased_projection_df = pd.DataFrame(projection_results)

new_order_candidates_df = new_order_candidates_df.merge(
    time_phased_projection_df,
    on="SKU_ID",
    how="left"
)

display(
    new_order_candidates_df[
        [
            "SKU_ID",
            "Closing_Inventory",
            "Future_Inbound_90D",
            "Forecast_90D_Units",
            "First_Projected_Shortage_Date",
            "Projected_Stock_31_Mar",
            "Supplier_Name",
            "Lead_Time_Days",
            "Supplier_Reliability"
        ]
    ]
    .sort_values("First_Projected_Shortage_Date", na_position="last")
    .head(20)
    .round(2)
)

In [ ]:
new_order_candidates_df["New_Order_Arrival_Date"] = (
    planning_date
    + pd.to_timedelta(new_order_candidates_df["Lead_Time_Days"], unit="D")
)

# Calculate forecast demand from 1 Jan through the new order arrival date
arrival_demand = []

for row in new_order_candidates_df.itertuples(index=False):
    sku = row.SKU_ID
    arrival_date = row.New_Order_Arrival_Date

    sku_forecast = daily_forecast_lookup[sku]

    demand_until_arrival = sku_forecast.loc[
        forecast_start:arrival_date
    ].sum()

    arrival_demand.append(demand_until_arrival)

new_order_candidates_df["Demand_Until_New_Order_Arrival"] = arrival_demand

# Include existing inbound expected by the new order arrival date
new_order_candidates_df["Inbound_Until_New_Order_Arrival"] = (
    new_order_candidates_df["Inbound_Within_Lead_Time"]
)

new_order_candidates_df["Stock_At_New_Order_Arrival"] = (
    new_order_candidates_df["Closing_Inventory"]
    + new_order_candidates_df["Inbound_Until_New_Order_Arrival"]
    - new_order_candidates_df["Demand_Until_New_Order_Arrival"]
)

display(
    new_order_candidates_df[
        [
            "SKU_ID",
            "Closing_Inventory",
            "Lead_Time_Days",
            "New_Order_Arrival_Date",
            "Demand_Until_New_Order_Arrival",
            "Inbound_Until_New_Order_Arrival",
            "Stock_At_New_Order_Arrival",
            "Supplier_Reliability"
        ]
    ]
    .sort_values("Stock_At_New_Order_Arrival")
    .head(20)
    .round(2)
)

In [ ]:
arrival_risk_results = []

for row in new_order_candidates_df.itertuples(index=False):
    sku = row.SKU_ID
    arrival_date = row.New_Order_Arrival_Date

    stock = float(row.Closing_Inventory)
    daily_demand = daily_forecast_lookup[sku]
    daily_receipts = receipt_lookup.get(sku, pd.Series(dtype=float))

    first_shortage_before_arrival = pd.NaT
    minimum_stock_before_arrival = stock

    for date in pd.date_range(forecast_start, arrival_date, freq="D"):
        stock += daily_receipts.get(date, 0)
        stock -= daily_demand.loc[date]

        minimum_stock_before_arrival = min(
            minimum_stock_before_arrival, stock
        )

        if stock < 0 and pd.isna(first_shortage_before_arrival):
            first_shortage_before_arrival = date

    arrival_risk_results.append({
        "SKU_ID": sku,
        "First_Shortage_Before_Arrival": first_shortage_before_arrival,
        "Minimum_Stock_Before_Arrival": minimum_stock_before_arrival,
        "Stock_At_Arrival": stock,
        "Lead_Time_Days": row.Lead_Time_Days,
        "New_Order_Arrival_Date": arrival_date
    })

arrival_risk_df = pd.DataFrame(arrival_risk_results)

display(
    arrival_risk_df
    .sort_values(
        ["First_Shortage_Before_Arrival", "Stock_At_Arrival"],
        na_position="last"
    )
    .head(20)
    .round(2)
)

In [ ]:
# Combine the daily shortage results with opening stock
arrival_risk_summary_df = arrival_risk_df.merge(
    new_order_candidates_df[
        ["SKU_ID", "Closing_Inventory", "Supplier_Reliability"]
    ],
    on="SKU_ID",
    how="left"
)

arrival_risk_summary_df["Arrival_Risk_Category"] = np.select(
    [
        arrival_risk_summary_df["First_Shortage_Before_Arrival"].isna(),
        arrival_risk_summary_df["First_Shortage_Before_Arrival"]
            <= forecast_start,
        arrival_risk_summary_df["First_Shortage_Before_Arrival"]
            < arrival_risk_summary_df["New_Order_Arrival_Date"]
    ],
    [
        "No Shortage Through Arrival",
        "Immediate Shortage - 1 Jan",
        "Shortage Before Arrival"
    ],
    default="Shortage On Arrival Date"
)

arrival_risk_summary = (
    arrival_risk_summary_df
    .groupby("Arrival_Risk_Category")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Zero_Opening_Stock_SKUs=(
            "Closing_Inventory",
            lambda x: (x == 0).sum()
        ),
        Opening_Inventory=("Closing_Inventory", "sum")
    )
    .reset_index()
)

display(arrival_risk_summary)

In [ ]:
# Merge the daily risk details with candidate information
purchase_priority_df = (
    new_order_candidates_df
    .merge(
        arrival_risk_df[
            [
                "SKU_ID",
                "First_Shortage_Before_Arrival",
                "Minimum_Stock_Before_Arrival",
                "Stock_At_Arrival"
            ]
        ],
        on="SKU_ID",
        how="left"
    )
)

# Days from planning date until first projected shortage
purchase_priority_df["Days_Until_Shortage"] = (
    purchase_priority_df["First_Shortage_Before_Arrival"]
    - planning_date
).dt.days

# Estimated shortage coverage needed by the new order arrival date
purchase_priority_df["Lead_Time_Shortfall_Units"] = (
    -purchase_priority_df["Stock_At_Arrival"]
).clip(lower=0)

# Assign urgency
purchase_priority_df["Purchase_Urgency"] = np.select(
    [
        purchase_priority_df["Days_Until_Shortage"].isna(),
        purchase_priority_df["Days_Until_Shortage"] <= 0,
        purchase_priority_df["Days_Until_Shortage"] <= 7,
        purchase_priority_df["Days_Until_Shortage"] <= 14
    ],
    [
        "No Shortage Through Arrival",
        "Immediate",
        "Urgent - Within 7 Days",
        "High - Within 14 Days"
    ],
    default="Normal Replenishment"
)

display(
    purchase_priority_df[
        [
            "SKU_ID",
            "Closing_Inventory",
            "First_Shortage_Before_Arrival",
            "Days_Until_Shortage",
            "New_Order_Arrival_Date",
            "Stock_At_Arrival",
            "Lead_Time_Shortfall_Units",
            "Purchase_Urgency",
            "Supplier_Name",
            "Supplier_Reliability"
        ]
    ]
    .sort_values(
        ["Days_Until_Shortage", "Lead_Time_Shortfall_Units"],
        ascending=[True, False],
        na_position="last"
    )
    .head(25)
    .round(2)
)

In [ ]:
arrival_risk_results = []

for row in new_order_candidates_df.itertuples(index=False):
    sku = row.SKU_ID
    arrival_date = row.New_Order_Arrival_Date

    stock = float(row.Closing_Inventory)
    daily_demand = daily_forecast_lookup[sku]
    daily_receipts = receipt_lookup.get(sku, pd.Series(dtype=float))

    first_shortage_date = pd.NaT
    minimum_stock = stock
    cumulative_unmet_demand = 0.0

    for date in pd.date_range(forecast_start, arrival_date, freq="D"):

        # Add receipts arriving on this date
        available_stock = stock + daily_receipts.get(date, 0)

        # Forecast demand for the date
        demand = float(daily_demand.loc[date])

        # Record the first date when available stock cannot cover demand
        if available_stock < demand and pd.isna(first_shortage_date):
            first_shortage_date = date

        # Estimate demand that cannot be fulfilled
        unmet_demand = max(0, demand - available_stock)
        cumulative_unmet_demand += unmet_demand

        # Physical inventory cannot fall below zero
        stock = max(0, available_stock - demand)
        minimum_stock = min(minimum_stock, stock)

    arrival_risk_results.append({
        "SKU_ID": sku,
        "First_Shortage_On_Or_Before_Arrival": first_shortage_date,
        "Minimum_Physical_Stock": minimum_stock,
        "Physical_Stock_At_Arrival": stock,
        "Estimated_Unmet_Demand_Through_Arrival": cumulative_unmet_demand,
        "Lead_Time_Days": row.Lead_Time_Days,
        "New_Order_Arrival_Date": arrival_date
    })

arrival_risk_df = pd.DataFrame(arrival_risk_results)

# Add opening inventory for the summary
arrival_risk_summary_df = arrival_risk_df.merge(
    new_order_candidates_df[
        ["SKU_ID", "Closing_Inventory", "Supplier_Reliability"]
    ],
    on="SKU_ID",
    how="left"
)

arrival_risk_summary_df["Days_Until_Shortage"] = (
    arrival_risk_summary_df["First_Shortage_On_Or_Before_Arrival"]
    - planning_date
).dt.days

arrival_risk_summary_df["Arrival_Risk_Category"] = np.select(
    [
        arrival_risk_summary_df["First_Shortage_On_Or_Before_Arrival"].isna(),
        arrival_risk_summary_df["Days_Until_Shortage"] <= 1,
        arrival_risk_summary_df["First_Shortage_On_Or_Before_Arrival"]
            < arrival_risk_summary_df["New_Order_Arrival_Date"]
    ],
    [
        "No Shortage Through Arrival",
        "Immediate Shortage - 1 Jan",
        "Shortage Before Arrival"
    ],
    default="Shortage On Arrival Date"
)

display(
    arrival_risk_summary_df.groupby("Arrival_Risk_Category")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Zero_Opening_Stock_SKUs=(
            "Closing_Inventory", lambda x: (x == 0).sum()
        ),
        Opening_Inventory=("Closing_Inventory", "sum"),
        Estimated_Unmet_Demand=(
            "Estimated_Unmet_Demand_Through_Arrival", "sum"
        )
    )
    .reset_index()
)

display(
    arrival_risk_summary_df.sort_values(
        ["Days_Until_Shortage", "Estimated_Unmet_Demand_Through_Arrival"],
        ascending=[True, False],
        na_position="last"
    )[
        [
            "SKU_ID",
            "Closing_Inventory",
            "First_Shortage_On_Or_Before_Arrival",
            "Days_Until_Shortage",
            "New_Order_Arrival_Date",
            "Physical_Stock_At_Arrival",
            "Estimated_Unmet_Demand_Through_Arrival",
            "Supplier_Reliability"
        ]
    ].head(25).round(2)
)

In [ ]:
bridge_stock_results = []

for row in new_order_candidates_df.itertuples(index=False):
    sku = row.SKU_ID
    arrival_date = row.New_Order_Arrival_Date

    opening_stock = float(row.Closing_Inventory)
    daily_demand = daily_forecast_lookup[sku]
    daily_receipts = receipt_lookup.get(sku, pd.Series(dtype=float))

    running_balance = opening_stock
    minimum_projected_balance = opening_stock
    date_of_maximum_shortage = pd.NaT

    for date in pd.date_range(forecast_start, arrival_date, freq="D"):
        running_balance += daily_receipts.get(date, 0)
        running_balance -= float(daily_demand.loc[date])

        if running_balance < minimum_projected_balance:
            minimum_projected_balance = running_balance
            date_of_maximum_shortage = date

    bridge_stock_required = max(0, -minimum_projected_balance)

    bridge_stock_results.append({
        "SKU_ID": sku,
        "Bridge_Stock_Required": bridge_stock_required,
        "Date_of_Maximum_Shortage": date_of_maximum_shortage,
        "Minimum_Projected_Balance": minimum_projected_balance
    })

bridge_stock_df = pd.DataFrame(bridge_stock_results)

# Combine with supplier and arrival details
bridge_stock_df = bridge_stock_df.merge(
    new_order_candidates_df[
        [
            "SKU_ID",
            "Category",
            "Closing_Inventory",
            "New_Order_Arrival_Date",
            "Lead_Time_Days",
            "Supplier_Name",
            "Supplier_Reliability"
        ]
    ],
    on="SKU_ID",
    how="left"
)

display(
    bridge_stock_df.sort_values(
        "Bridge_Stock_Required",
        ascending=False
    ).head(25).round(2)
)

print(
    "Total estimated bridge stock required:",
    round(bridge_stock_df["Bridge_Stock_Required"].sum(), 2)
)

In [ ]:
# Get alternate supplier mappings for the 134 candidate SKUs
alternate_supplier_df = (
    product_supplier_df[
        product_supplier_df["Supplier_Role"] != "Preferred"
    ]
    .merge(
        suppliers_df,
        on="Supplier_ID",
        how="left"
    )
)

# Add preferred supplier details for comparison
preferred_comparison_df = (
    new_order_candidates_df[
        [
            "SKU_ID",
            "Supplier_ID",
            "Supplier_Name",
            "Lead_Time_Days",
            "Supplier_Reliability"
        ]
    ]
    .rename(columns={
        "Supplier_ID": "Preferred_Supplier_ID",
        "Supplier_Name": "Preferred_Supplier_Name",
        "Lead_Time_Days": "Preferred_Lead_Time_Days",
        "Supplier_Reliability": "Preferred_Supplier_Reliability"
    })
)

# Compare each alternate supplier with the preferred supplier
alternate_supplier_options_df = alternate_supplier_df.merge(
    preferred_comparison_df,
    on="SKU_ID",
    how="inner"
)

alternate_supplier_options_df["Lead_Time_Saving_Days"] = (
    alternate_supplier_options_df["Preferred_Lead_Time_Days"]
    - alternate_supplier_options_df["Lead_Time_Days"]
)

# Keep the useful comparison fields
alternate_supplier_options_df = alternate_supplier_options_df[
    [
        "SKU_ID",
        "Supplier_Role",
        "Supplier_ID",
        "Supplier_Name",
        "Lead_Time_Days",
        "Supplier_Reliability",
        "Preferred_Supplier_Name",
        "Preferred_Lead_Time_Days",
        "Preferred_Supplier_Reliability",
        "Lead_Time_Saving_Days"
    ]
].sort_values(
    ["SKU_ID", "Lead_Time_Days"]
)

display(alternate_supplier_options_df.head(30))

In [ ]:
display(
    product_supplier_df["Supplier_Role"]
    .value_counts()
    .rename_axis("Supplier_Role")
    .reset_index(name="SKU_Mapping_Count")
)

In [ ]:
# Calculate the 90-day replenishment requirement

new_order_candidates_df["Safety_Stock_Units"] = 2

new_order_candidates_df["Net_90D_Requirement"] = (
    new_order_candidates_df["Forecast_90D_Units"]
    + new_order_candidates_df["Safety_Stock_Units"]
    - new_order_candidates_df["Closing_Inventory"]
    - new_order_candidates_df["Future_Inbound_90D"]
)

new_order_candidates_df["Recommended_Order_Quantity"] = (
    new_order_candidates_df["Net_90D_Requirement"]
    .clip(lower=0)
    .apply(np.ceil)
    .astype(int)
)

# Summarise recommendations
purchase_recommendation_summary = (
    new_order_candidates_df
    .groupby("Category")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Forecast_90D_Units=("Forecast_90D_Units", "sum"),
        Closing_Inventory=("Closing_Inventory", "sum"),
        Future_Inbound_90D=("Future_Inbound_90D", "sum"),
        Recommended_Order_Quantity=("Recommended_Order_Quantity", "sum")
    )
    .reset_index()
)

display(purchase_recommendation_summary.round(2))

print(
    "SKUs with a positive recommended order:",
    (new_order_candidates_df["Recommended_Order_Quantity"] > 0).sum()
)

print(
    "Total recommended order quantity:",
    new_order_candidates_df["Recommended_Order_Quantity"].sum()
)

In [ ]:
# Start with all 1,100 SKUs
full_sku_purchase_plan_df = sku_stock_position_df.copy()

# Add 90-day forecast demand
full_sku_purchase_plan_df = full_sku_purchase_plan_df.merge(
    forecast_90d_by_sku,
    on="SKU_ID",
    how="left"
)

# Add future inbound expected within the forecast period
full_sku_purchase_plan_df = full_sku_purchase_plan_df.merge(
    future_inbound_90d_by_sku,
    on="SKU_ID",
    how="left"
)

# Replace missing inbound with zero
full_sku_purchase_plan_df["Future_Inbound_90D"] = (
    full_sku_purchase_plan_df["Future_Inbound_90D"].fillna(0)
)

# Calculate purchase requirement
full_sku_purchase_plan_df["Safety_Stock_Units"] = 2

full_sku_purchase_plan_df["Net_90D_Requirement"] = (
    full_sku_purchase_plan_df["Forecast_90D_Units"]
    + full_sku_purchase_plan_df["Safety_Stock_Units"]
    - full_sku_purchase_plan_df["Closing_Inventory"]
    - full_sku_purchase_plan_df["Future_Inbound_90D"]
)

full_sku_purchase_plan_df["Recommended_Order_Quantity"] = (
    full_sku_purchase_plan_df["Net_90D_Requirement"]
    .clip(lower=0)
    .apply(np.ceil)
    .astype(int)
)

# Category-level summary
full_purchase_summary_df = (
    full_sku_purchase_plan_df
    .groupby("Category")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        Forecast_90D_Units=("Forecast_90D_Units", "sum"),
        Closing_Inventory=("Closing_Inventory", "sum"),
        Future_Inbound_90D=("Future_Inbound_90D", "sum"),
        Recommended_Order_Quantity=(
            "Recommended_Order_Quantity", "sum"
        )
    )
    .reset_index()
)

display(full_purchase_summary_df.round(2))

print("Total SKUs:", full_sku_purchase_plan_df["SKU_ID"].nunique())
print(
    "SKUs requiring purchase:",
    (full_sku_purchase_plan_df["Recommended_Order_Quantity"] > 0).sum()
)
print(
    "Total recommended purchase quantity:",
    full_sku_purchase_plan_df["Recommended_Order_Quantity"].sum()
)

In [ ]:
# Summarise purchasing recommendations by lead-time risk
# and purchase-order exception status

purchase_risk_summary_df = (
    full_sku_purchase_plan_df
    .groupby(
        ["Lead_Time_Risk", "PO_Exception_Status"],
        dropna=False
    )
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        SKUs_Requiring_Purchase=(
            "Recommended_Order_Quantity",
            lambda x: (x > 0).sum()
        ),
        Recommended_Order_Quantity=(
            "Recommended_Order_Quantity", "sum"
        ),
        Overdue_PO_Units=("Overdue_PO_Units", "sum"),
        Due_On_Cutoff_Units=("Due_On_Cutoff_Units", "sum"),
        Future_Inbound_90D=("Future_Inbound_90D", "sum")
    )
    .reset_index()
)

display(purchase_risk_summary_df)

In [ ]:
import numpy as np

full_sku_purchase_plan_df["Purchasing_Action"] = np.select(
    [
        (full_sku_purchase_plan_df["Lead_Time_Risk"] == "Below Safety Stock") &
        (full_sku_purchase_plan_df["PO_Exception_Status"] == "Has Overdue PO"),

        (full_sku_purchase_plan_df["Lead_Time_Risk"] == "Below Safety Stock") &
        (full_sku_purchase_plan_df["PO_Exception_Status"] == "Due on Planning Date"),

        (full_sku_purchase_plan_df["Lead_Time_Risk"] == "Below Safety Stock") &
        (full_sku_purchase_plan_df["PO_Exception_Status"] == "No Overdue or Due PO"),

        full_sku_purchase_plan_df["PO_Exception_Status"].isin(
            ["Has Overdue PO", "Due on Planning Date"]
        ),

        full_sku_purchase_plan_df["Recommended_Order_Quantity"] > 0
    ],
    [
        "Immediate Risk - Overdue PO",
        "Immediate Risk - PO Due",
        "Immediate Risk - New Order",
        "Supplier Confirmation",
        "Routine Replenishment"
    ],
    default="No Purchase Required"
)

purchasing_action_summary_df = (
    full_sku_purchase_plan_df
    .groupby("Purchasing_Action")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        SKUs_With_Purchase_Requirement=(
            "Recommended_Order_Quantity",
            lambda x: (x > 0).sum()
        ),
        Planning_Order_Quantity=(
            "Recommended_Order_Quantity", "sum"
        ),
        Overdue_PO_Units=("Overdue_PO_Units", "sum"),
        Due_On_Cutoff_Units=("Due_On_Cutoff_Units", "sum"),
        Future_Inbound_90D=("Future_Inbound_90D", "sum")
    )
    .reset_index()
    .sort_values("SKU_Count", ascending=False)
)

display(purchasing_action_summary_df)


In [ ]:
purchasing_action_detail_df = (
    full_sku_purchase_plan_df[
        [
            "SKU_ID",
            "Category",
            "Purchasing_Action",
            "Lead_Time_Risk",
            "PO_Exception_Status",
            "Closing_Inventory",
            "Forecast_90D_Units",
            "Future_Inbound_90D",
            "Overdue_PO_Units",
            "Due_On_Cutoff_Units",
            "Lead_Time_Days",
            "Supplier_Name",
            "Supplier_Reliability",
            "Recommended_Order_Quantity"
        ]
    ]
    .copy()
    .sort_values(
        ["Purchasing_Action", "Recommended_Order_Quantity"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(purchasing_action_detail_df.head(15))

print("Total SKUs:", purchasing_action_detail_df["SKU_ID"].nunique())
print(
    "Total planning order quantity:",
    purchasing_action_detail_df["Recommended_Order_Quantity"].sum()
)

In [ ]:
print("Purchasing action summary:", purchasing_action_summary_df.shape)
print("Purchasing detail:", purchasing_action_detail_df.shape)
print("Category summary:", full_sku_purchase_plan_df.groupby("Category")["Recommended_Order_Quantity"].sum().shape)
print("Open PO exceptions:", outstanding_po_detail.shape)
print("Forecast summary:", category_forecast_summary_df.shape if "category_forecast_summary_df" in globals() else "Not found")


In [ ]:
print("Forecast columns:")
print(final_sku_forecasts_df.columns.tolist())

print("\nProduct columns:")
print(products_df.columns.tolist())

print("\nPurchasing plan columns:")
print(full_sku_purchase_plan_df.columns.tolist())

In [ ]:
# 1. Category-level purchasing summary
category_purchase_summary_df = (
    full_sku_purchase_plan_df
    .groupby("Category")
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        SKUs_Requiring_Purchase=(
            "Recommended_Order_Quantity",
            lambda x: (x > 0).sum()
        ),
        Closing_Inventory=("Closing_Inventory", "sum"),
        Forecast_90D_Units=("Forecast_90D_Units", "sum"),
        Future_Inbound_90D=("Future_Inbound_90D", "sum"),
        Recommended_Order_Quantity=(
            "Recommended_Order_Quantity", "sum"
        )
    )
    .reset_index()
)

# 2. Supplier-level purchasing summary
supplier_purchase_summary_df = (
    full_sku_purchase_plan_df
    .groupby(
        ["Supplier_ID", "Supplier_Name", "Supplier_Reliability"],
        dropna=False
    )
    .agg(
        SKU_Count=("SKU_ID", "nunique"),
        SKUs_Requiring_Purchase=(
            "Recommended_Order_Quantity",
            lambda x: (x > 0).sum()
        ),
        Forecast_90D_Units=("Forecast_90D_Units", "sum"),
        Closing_Inventory=("Closing_Inventory", "sum"),
        Future_Inbound_90D=("Future_Inbound_90D", "sum"),
        Overdue_PO_Units=("Overdue_PO_Units", "sum"),
        Recommended_Order_Quantity=(
            "Recommended_Order_Quantity", "sum"
        )
    )
    .reset_index()
    .sort_values("Recommended_Order_Quantity", ascending=False)
)

# 3. Category-level forecast comparison
category_forecast_summary_df = (
    final_sku_forecasts_df
    .groupby("Category")
    .agg(
        Moving_Average_90D=("Moving_Average", "sum"),
        Seasonal_Naive_90D=("Seasonal_Naive", "sum"),
        Exponential_Smoothing_90D=(
            "Exponential_Smoothing", "sum"
        )
    )
    .reset_index()
)

print("CATEGORY PURCHASE SUMMARY")
display(category_purchase_summary_df)

print("SUPPLIER PURCHASE SUMMARY")
display(supplier_purchase_summary_df)

print("CATEGORY FORECAST SUMMARY")
display(category_forecast_summary_df)

In [ ]:
from pathlib import Path

output_folder = Path(
    r"C:\Business\06_Projects\MarketPulse_Retail_Demand_Forecasting\data\final"
)
output_folder.mkdir(parents=True, exist_ok=True)

excel_output_path = output_folder / "MarketPulse_Forecasting_and_Purchasing_Analysis.xlsx"

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:

    purchasing_action_detail_df.to_excel(
        writer, sheet_name="Purchasing_Action_Detail", index=False
    )

    purchasing_action_summary_df.to_excel(
        writer, sheet_name="Purchasing_Action_Summary", index=False
    )

    category_purchase_summary_df.to_excel(
        writer, sheet_name="Category_Purchase_Summary", index=False
    )

    supplier_purchase_summary_df.to_excel(
        writer, sheet_name="Supplier_Summary", index=False
    )

    outstanding_po_detail.to_excel(
        writer, sheet_name="Open_PO_Exceptions", index=False
    )

    category_forecast_summary_df.to_excel(
        writer, sheet_name="Forecast_Summary", index=False
    )

    sku_stock_position_df.to_excel(
        writer, sheet_name="SKU_Stock_Position", index=False
    )

print("Excel workbook created successfully:")
print(excel_output_path)

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb = load_workbook(excel_output_path)

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    # Format header row
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    ws.row_dimensions[1].height = 32

    # Set practical column widths based on content
    for column_cells in ws.columns:
        column_letter = get_column_letter(column_cells[0].column)

        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in column_cells[:200]
        )

        ws.column_dimensions[column_letter].width = min(
            max(max_length + 2, 12),
            32
        )

wb.save(excel_output_path)

print("Workbook formatting completed.")
print("Sheets:", wb.sheetnames)


In [ ]:
from openpyxl import load_workbook

check_wb = load_workbook(excel_output_path, read_only=True, data_only=True)

for ws in check_wb.worksheets:
    print(
        f"{ws.title}: "
        f"{ws.max_row - 1:,} data rows, "
        f"{ws.max_column} columns"
    )

check_wb.close()

print("\nFile size (MB):", round(excel_output_path.stat().st_size / (1024 ** 2), 2))

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# Add daily SKU-level forecast data
with pd.ExcelWriter(
    excel_output_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    final_sku_forecasts_df.to_excel(
        writer,
        sheet_name="Daily_SKU_Forecast",
        index=False
    )

# Format the new sheet
wb = load_workbook(excel_output_path)
ws = wb["Daily_SKU_Forecast"]

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill(fill_type="solid", fgColor="1F4E78")
    cell.alignment = Alignment(horizontal="center", vertical="center")

ws.row_dimensions[1].height = 28

for column_cells in ws.columns:
    letter = get_column_letter(column_cells[0].column)
    max_length = max(
        len(str(cell.value)) if cell.value is not None else 0
        for cell in column_cells[:200]
    )
    ws.column_dimensions[letter].width = min(max(max_length + 2, 12), 24)

# Date and forecast number formats
for cell in ws["A"][1:]:
    cell.number_format = "dd-mmm-yyyy"

for col in ["C", "D", "E"]:
    for cell in ws[col][1:]:
        cell.number_format = "#,##0.00"

wb.save(excel_output_path)
wb.close()

print("Daily SKU forecast sheet added.")
print("Records:", len(final_sku_forecasts_df))